# DATA 266 — HW12 | Knowledge Distillation (SEED = 1598)

**Submission notebook — Part 2 and its separate distillation measurements/analysis.** The GPTQ experiment is maintained in `DATA266_HW12_GPTQ.ipynb` and is **not included or altered here**.

**Objective:** distill the locally provided AG News BERT teacher into `distilbert/distilbert-base-uncased`, compare controlled temperature and α sweeps, then evaluate the selected student against the teacher.

**Evidence policy:** the original executed code, cell outputs, and figures from the uploaded notebook are retained. Outdated seed-42 runs and GPTQ planning cells have been excluded from this submission copy. Section headings and explanations are streamlined; the original notebook is preserved separately.

**Reproduction guidance:** this notebook documents the historical training run and references saved checkpoints under Google Drive. **Do not click “Run all” against the populated experiment folders.** Training cells intentionally refuse to overwrite previous checkpoints. For a fresh reproduction, use an isolated, empty output workspace and check its paths before training. Read-only evaluation cells may be rerun after the required model/data setup. Earlier outputs may show a different runtime than final Tesla T4 benchmarks.

## Results at a glance

| Model | AG News test accuracy | Macro-F1 | p50 / p95 inference (ms) | Peak allocated GPU (GiB) | Checkpoint directory size (MiB) |
|---|---:|---:|---:|---:|---:|
| BERT teacher | 94.4605% | 0.944652 | 16.7124 / 17.1056 | 0.4264 | 418.5823 |
| DistilBERT student (T=5, α=0.25) | **94.6579%** | **0.946626** | **8.5814 / 8.9077** | **0.2680** | **255.4317** |

The student improved accuracy by **0.1974 percentage points** on this test, and its p50 forward-pass latency was approximately **1.95× faster** in this benchmark. These are measurements from the corrected seed-1598 experiment, not the older seed-42 run. GPU memory is PyTorch peak **allocated** memory, not total device usage; sizes are binary MiB/GiB.

## 01 | Runtime, seed and project paths

Mount Drive and configure personal seed 1598. The GPU shown in the final evaluation was a Tesla T4; earlier experiments may have been run on a different runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# DATA 266 HW12 — PERSONAL PARAMETERS AND SAFE OUTPUT PATHS
# ============================================================

import random
import numpy as np
import torch
from pathlib import Path

SID4 = 1598
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path("/content/drive/MyDrive/DATA266_HW12")

# OLD RESULTS — READ ONLY
OLD_PART2 = ROOT / "outputs/part2_distillation"
OLD_PART3 = ROOT / "outputs/part3_analysis"

# NEW RESULTS — SEED 1598
NEW_PART2 = ROOT / "outputs/part2_distillation_seed1598"
NEW_PART3 = ROOT / "outputs/part3_analysis_seed1598"

assert OLD_PART2.is_dir(), "Original Part 2 results missing."
assert OLD_PART3.is_dir(), "Original Part 3 results missing."

assert NEW_PART2 != OLD_PART2
assert NEW_PART3 != OLD_PART3

NEW_PART2.mkdir(parents=True, exist_ok=True)
NEW_PART3.mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("DATA266 HW12 — CORRECTED PART 2 SETUP")
print("=" * 65)

print("SID4:", SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)

print("\nOriginal results:")
print("Part 2:", OLD_PART2)
print("Part 3:", OLD_PART3)

print("\nCorrected results:")
print("Part 2:", NEW_PART2)
print("Part 3:", NEW_PART3)

print("\nGPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n✅ Personal parameters configured.")
print("✅ Separate output directories created.")
print("✅ Original experiment directories untouched.")

print("=" * 65)

DATA266 HW12 — CORRECTED PART 2 SETUP
SID4: 1598
SEED: 1598
SLICE: 598
HP_ID: 2
CLS_A: 8
CLS_B: 5

Original results:
Part 2: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation
Part 3: /content/drive/MyDrive/DATA266_HW12/outputs/part3_analysis

Corrected results:
Part 2: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598
Part 3: /content/drive/MyDrive/DATA266_HW12/outputs/part3_analysis_seed1598

GPU available: True
GPU: NVIDIA L4

✅ Personal parameters configured.
✅ Separate output directories created.
✅ Original experiment directories untouched.


In [3]:

from pathlib import Path
import torch

SID4 = 1598
SEED = 1598

ROOT = Path("/content/drive/MyDrive/DATA266_HW12")
NEW_PART2 = ROOT / "outputs/part2_distillation_seed1598"

assert NEW_PART2.is_dir()
assert torch.cuda.is_available(), "Connect to a GPU runtime"

print("Seed:", SEED)
print("GPU:", torch.cuda.get_device_name(0))
print("Output directory:", NEW_PART2)
print("Ready for final alpha-sweep comparison.")


Seed: 1598
GPU: Tesla T4
Output directory: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598
Ready for final alpha-sweep comparison.


## 02 | Teacher checkpoint and AG News dataset

Check the instructor-provided BERT checkpoint, load AG News, and make the stratified 108,000/12,000 train/validation split. The official 7,600-example test set is held out until final evaluation.

In [ ]:
from pathlib import Path

teacher_path = Path(
    "/content/drive/MyDrive/DATA266_HW12/teacher_model"
)

print("TEACHER MODEL FILES")
print("=" * 55)

for file in sorted(teacher_path.iterdir()):
    if file.is_file():
        size_mb = file.stat().st_size / (1024 ** 2)
        print(f"{file.name:<40} {size_mb:.2f} MB")
    else:
        print(f"[FOLDER] {file.name}")

print("=" * 55)

config_found = (teacher_path / "config.json").is_file()

weights_found = (
    any(teacher_path.glob("*.safetensors"))
    or any(teacher_path.glob("pytorch_model*.bin"))
)

print("Config found:", config_found)
print("Model weights found:", weights_found)

TEACHER MODEL FILES
[FOLDER] __MACOSX
config.json                              0.00 MB
model.safetensors                        417.68 MB
special_tokens_map.json                  0.00 MB
tokenizer.json                           0.68 MB
tokenizer_config.json                    0.00 MB
vocab.txt                                0.22 MB
Config found: True
Model weights found: True


In [ ]:
import json
from pathlib import Path

TEACHER_PATH = Path(
    "/content/drive/MyDrive/DATA266_HW12/teacher_model"
)

with open(TEACHER_PATH / "config.json", "r") as f:
    config = json.load(f)

print("=" * 55)
print("TEACHER MODEL CONFIGURATION")
print("=" * 55)

for key in [
    "model_type",
    "architectures",
    "num_labels",
    "id2label",
    "label2id",
    "hidden_size",
    "num_hidden_layers",
    "vocab_size"
]:
    print(f"{key}: {config.get(key, 'Not specified')}")

print("=" * 55)

TEACHER MODEL CONFIGURATION
model_type: bert
architectures: ['BertForSequenceClassification']
num_labels: Not specified
id2label: {'0': 'WORLD', '1': 'SPORTS', '2': 'BUSINESS', '3': 'SCI/TECH'}
label2id: {'LABEL_0': 0, 'LABEL_1': 1, 'LABEL_2': 2, 'LABEL_3': 3}
hidden_size: 768
num_hidden_layers: 12
vocab_size: 30522


In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

TEACHER_DIR = "/content/drive/MyDrive/DATA266_HW12/teacher_model"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Loading teacher tokenizer...")

teacher_tokenizer = AutoTokenizer.from_pretrained(
    TEACHER_DIR,
    local_files_only=True
)

print("Loading teacher model...")

teacher_model = AutoModelForSequenceClassification.from_pretrained(
    TEACHER_DIR,
    local_files_only=True
)

teacher_model = teacher_model.to(device)
teacher_model.eval()

print("\n" + "=" * 55)
print("TEACHER MODEL VERIFICATION")
print("=" * 55)

print("Device:", device)
print("Model type:", teacher_model.config.model_type)
print("Number of classes:", teacher_model.config.num_labels)
print("Class labels:", teacher_model.config.id2label)

total_params = sum(
    p.numel() for p in teacher_model.parameters()
)

print(f"Total parameters: {total_params:,}")

print("=" * 55)
print("Teacher model loaded successfully!")

Loading teacher tokenizer...
Loading teacher model...

TEACHER MODEL VERIFICATION
Device: cuda
Model type: bert
Number of classes: 4
Class labels: {0: 'WORLD', 1: 'SPORTS', 2: 'BUSINESS', 3: 'SCI/TECH'}
Total parameters: 109,485,316
Teacher model loaded successfully!


In [ ]:

from datasets import load_dataset

print("=" * 65)
print("LOADING AG NEWS — SEED 1598 EXPERIMENT")
print("=" * 65)

# Load the same canonical AG News dataset.
ag_news = load_dataset("fancyzhx/ag_news")

# Verify the expected dataset structure.
assert len(ag_news["train"]) == 120000
assert len(ag_news["test"]) == 7600

assert "text" in ag_news["train"].column_names
assert "label" in ag_news["train"].column_names

print("Dataset loaded successfully!")
print("Training examples:", len(ag_news["train"]))
print("Test examples:", len(ag_news["test"]))
print("Columns:", ag_news["train"].column_names)

print("\n✅ AG News is ready.")
print("✅ No training performed.")
print("✅ No checkpoints modified.")
print("=" * 65)


LOADING AG NEWS — SEED 1598 EXPERIMENT


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Dataset loaded successfully!
Training examples: 120000
Test examples: 7600
Columns: ['text', 'label']

✅ AG News is ready.
✅ No training performed.
✅ No checkpoints modified.


In [ ]:

import json
import random
import numpy as np
import torch

from pathlib import Path
from sklearn.model_selection import train_test_split

# ============================================================
# 1. REQUIRED PERSONAL SEED
# ============================================================

assert SID4 == 1598
assert SEED == SID4 == 1598

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# 2. ORIGINAL AG NEWS TRAINING DATA
# ============================================================

assert "ag_news" in globals(), (
    "Load AG News before running this cell."
)

assert len(ag_news["train"]) == 120000
assert len(ag_news["test"]) == 7600

all_labels = np.array(ag_news["train"]["label"])
all_indices = np.arange(len(ag_news["train"]))

# ============================================================
# 3. STRATIFIED SPLIT WITH SEED 1598
# ============================================================

train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=SEED,
    stratify=all_labels
)

train_data = ag_news["train"].select(
    train_indices.tolist()
)

val_data = ag_news["train"].select(
    val_indices.tolist()
)

test_data = ag_news["test"]

# ============================================================
# 4. VALIDATE THE SPLIT
# ============================================================

assert len(train_data) == 108000
assert len(val_data) == 12000
assert len(test_data) == 7600

assert len(set(train_indices) & set(val_indices)) == 0

assert (
    len(set(train_indices) | set(val_indices))
    == 120000
)

labels = ["WORLD", "SPORTS", "BUSINESS", "SCI/TECH"]

print("=" * 65)
print("KNOWLEDGE DISTILLATION — CORRECTED DATA SPLIT")
print("=" * 65)

print("Seed:", SEED)
print(f"Training:   {len(train_data):,}")
print(f"Validation: {len(val_data):,}")
print(f"Test:       {len(test_data):,}")

print("\nClass distribution:")

for name, indices in [
    ("Train", train_indices),
    ("Validation", val_indices)
]:
    counts = np.bincount(
        all_labels[indices],
        minlength=4
    )

    print(f"\n{name}:")

    for label, count in zip(labels, counts):
        print(f"  {label:<10} {count:,}")

print("\nTest:")

test_counts = np.bincount(
    np.array(test_data["label"]),
    minlength=4
)

for label, count in zip(labels, test_counts):
    print(f"  {label:<10} {count:,}")

# ============================================================
# 5. SAVE ONLY TO THE NEW OUTPUT DIRECTORY
# ============================================================

output_dir = NEW_PART2

assert output_dir == (
    Path("/content/drive/MyDrive/DATA266_HW12")
    / "outputs/part2_distillation_seed1598"
)

output_dir.mkdir(parents=True, exist_ok=True)

manifest_path = output_dir / "data_split_manifest.json"

split_manifest = {
    "dataset": "ag_news",
    "seed": SEED,
    "method": "sklearn_stratified_train_test_split",
    "train_size": len(train_data),
    "validation_size": len(val_data),
    "test_size": len(test_data),
    "train_original_indices": train_indices.tolist(),
    "validation_original_indices": val_indices.tolist()
}

# Protect an existing corrected manifest from being
# silently replaced by a different data split.
if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        existing_manifest = json.load(f)

    assert existing_manifest == split_manifest, (
        "Existing seed-1598 manifest differs. "
        "Stop and inspect before proceeding."
    )

    print("\n✅ Existing identical manifest verified.")

else:
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(split_manifest, f, indent=2)

    print("\n✅ New seed-1598 split manifest saved.")

print("Location:", manifest_path)

print("\n✅ Training and validation sets are disjoint.")
print("✅ Original test set remains separate.")
print("✅ Original seed-42 output folder untouched.")

print("=" * 65)


KNOWLEDGE DISTILLATION — CORRECTED DATA SPLIT
Seed: 1598
Training:   108,000
Validation: 12,000
Test:       7,600

Class distribution:

Train:
  WORLD      27,000
  SPORTS     27,000
  BUSINESS   27,000
  SCI/TECH   27,000

Validation:
  WORLD      3,000
  SPORTS     3,000
  BUSINESS   3,000
  SCI/TECH   3,000

Test:
  WORLD      1,900
  SPORTS     1,900
  BUSINESS   1,900
  SCI/TECH   1,900

✅ New seed-1598 split manifest saved.
Location: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/data_split_manifest.json

✅ Training and validation sets are disjoint.
✅ Original test set remains separate.
✅ Original seed-42 output folder untouched.


## 03 | Tokenization and teacher logits

Verify matching BERT/DistilBERT token IDs and cache teacher logits for the 108,000 training rows. These logits are aligned to `train_original_indices` in the saved split manifest.

In [ ]:

import hashlib
import json
from pathlib import Path

from transformers import AutoTokenizer

# ============================================================
# 1. CONFIGURATION
# ============================================================

assert SEED == SID4 == 1598
assert len(train_data) == 108000
assert len(val_data) == 12000

MAX_LENGTH = 256

TEACHER_DIR = ROOT / "teacher_model"
STUDENT_NAME = "distilbert/distilbert-base-uncased"

assert TEACHER_DIR.is_dir()
assert (TEACHER_DIR / "config.json").exists()

# ============================================================
# 2. LOAD TOKENIZERS
# ============================================================

print("Loading BERT teacher tokenizer...")

teacher_tokenizer = AutoTokenizer.from_pretrained(
    str(TEACHER_DIR),
    local_files_only=True
)

print("Loading DistilBERT student tokenizer...")

student_tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_NAME
)

# ============================================================
# 3. VERIFY TOKENIZER COMPATIBILITY
# ============================================================

sample_texts = [
    "The stock market increased today.",
    "The football team won the championship.",
    "Scientists discovered a new technology."
]

teacher_ids = teacher_tokenizer(
    sample_texts,
    truncation=True,
    max_length=MAX_LENGTH
)["input_ids"]

student_ids = student_tokenizer(
    sample_texts,
    truncation=True,
    max_length=MAX_LENGTH
)["input_ids"]

assert teacher_ids == student_ids, (
    "Teacher and student tokenization differ. "
    "Stop before generating teacher logits."
)

print("✅ Teacher/student tokenizers are compatible.")

# ============================================================
# 4. TOKENIZE THE CORRECTED DATASETS
# ============================================================

def tokenize_examples(batch):
    return student_tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

print("\nTokenizing corrected training dataset...")

tokenized_train = train_data.map(
    tokenize_examples,
    batched=True,
    batch_size=1000,
    desc="Seed 1598 — Train tokenization"
)

print("\nTokenizing corrected validation dataset...")

tokenized_val = val_data.map(
    tokenize_examples,
    batched=True,
    batch_size=1000,
    desc="Seed 1598 — Validation tokenization"
)

# ============================================================
# 5. VERIFY DATASET ALIGNMENT
# ============================================================

assert len(tokenized_train) == 108000
assert len(tokenized_val) == 12000

assert tokenized_train["label"] == train_data["label"]
assert tokenized_val["label"] == val_data["label"]

for idx in [0, 1, 100, 1000, 107999]:
    expected = student_tokenizer(
        train_data[idx]["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )["input_ids"]

    assert tokenized_train[idx]["input_ids"] == expected

# ============================================================
# 6. SAVE TOKENIZATION METADATA
# ============================================================

manifest_path = NEW_PART2 / "data_split_manifest.json"

with open(manifest_path, encoding="utf-8") as f:
    split_manifest = json.load(f)

assert split_manifest["seed"] == 1598
assert split_manifest["train_size"] == 108000

split_sha256 = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

metadata = {
    "seed": SEED,
    "teacher_tokenizer": str(TEACHER_DIR),
    "student_tokenizer": STUDENT_NAME,
    "max_length": MAX_LENGTH,
    "training_examples": len(tokenized_train),
    "validation_examples": len(tokenized_val),
    "train_dataset_fingerprint":
        tokenized_train._fingerprint,
    "validation_dataset_fingerprint":
        tokenized_val._fingerprint,
    "split_manifest_sha256": split_sha256,
    "training_order": "seed1598_split_manifest_order",
    "tokenizer_compatibility": "verified"
}

metadata_path = NEW_PART2 / "tokenization_metadata.json"

if metadata_path.exists():
    existing = json.loads(
        metadata_path.read_text(encoding="utf-8")
    )

    assert existing == metadata, (
        "Existing tokenization metadata differs. "
        "Inspect before overwriting."
    )
else:
    metadata_path.write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8"
    )

# ============================================================
# 7. SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("SEED-1598 TOKENIZATION VERIFICATION")
print("=" * 65)

print("Seed:", SEED)
print("Maximum token length:", MAX_LENGTH)
print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_val))
print("Tokenizer compatibility: PASSED")
print("Training alignment: PASSED")
print("Validation alignment: PASSED")

print("\nFirst training article:")
print("Text:", train_data[0]["text"][:150])
print("Label:", train_data[0]["label"])
print(
    "First 20 token IDs:",
    tokenized_train[0]["input_ids"][:20]
)

print("\nMetadata saved:", metadata_path)

print("\n✅ Seed-1598 tokenization complete.")
print("✅ No model training performed.")
print("✅ Original seed-42 files untouched.")

print("=" * 65)


Loading BERT teacher tokenizer...
Loading DistilBERT student tokenizer...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

✅ Teacher/student tokenizers are compatible.

Tokenizing corrected training dataset...


Seed 1598 — Train tokenization:   0%|          | 0/108000 [00:00<?, ? examples/s]


Tokenizing corrected validation dataset...


Seed 1598 — Validation tokenization:   0%|          | 0/12000 [00:00<?, ? examples/s]


SEED-1598 TOKENIZATION VERIFICATION
Seed: 1598
Maximum token length: 256
Training examples: 108000
Validation examples: 12000
Tokenizer compatibility: PASSED
Training alignment: PASSED
Validation alignment: PASSED

First training article:
Text: EU go-ahead for Oracle takeover The European Commission gives US software giant Oracle the green light for its takeover of US rival Peoplesoft.
Label: 2
First 20 token IDs: [101, 7327, 2175, 1011, 3805, 2005, 14721, 15336, 1996, 2647, 3222, 3957, 2149, 4007, 5016, 14721, 1996, 2665, 2422, 2005]

Metadata saved: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/tokenization_metadata.json

✅ Seed-1598 tokenization complete.
✅ No model training performed.
✅ Original seed-42 files untouched.


In [ ]:

import json
import torch
from pathlib import Path
from transformers import AutoModelForSequenceClassification

# ============================================================
# 1. SAFETY CHECKS
# ============================================================

assert SEED == SID4 == 1598
assert len(tokenized_train) == 108000
assert len(tokenized_val) == 12000

TEACHER_DIR = ROOT / "teacher_model"

assert (TEACHER_DIR / "config.json").is_file()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 65)
print("SEED-1598 — TEACHER MODEL VERIFICATION")
print("=" * 65)

# ============================================================
# 2. LOAD THE ORIGINAL FINE-TUNED TEACHER
# ============================================================

print("\nLoading existing BERT teacher checkpoint...")

teacher_model = (
    AutoModelForSequenceClassification.from_pretrained(
        str(TEACHER_DIR),
        local_files_only=True
    )
)

teacher_model = teacher_model.to(device)
teacher_model.eval()

for parameter in teacher_model.parameters():
    parameter.requires_grad_(False)

# ============================================================
# 3. CHECK MODEL CONFIGURATION
# ============================================================

print("\nTeacher model type:", teacher_model.config.model_type)
print("Number of classes:", teacher_model.config.num_labels)
print("Device:", device)

assert teacher_model.config.num_labels == 4

teacher_params = sum(
    p.numel() for p in teacher_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in teacher_model.parameters()
    if p.requires_grad
)

assert trainable_params == 0
assert not teacher_model.training

# ============================================================
# 4. TEST INFERENCE ON CORRECTED TRAINING ARTICLES
# ============================================================

sample_texts = [
    train_data[i]["text"]
    for i in range(4)
]

sample_labels = [
    train_data[i]["label"]
    for i in range(4)
]

encoded = teacher_tokenizer(
    sample_texts,
    padding=True,
    truncation=True,
    max_length=256,
    return_tensors="pt"
)

encoded = {
    key: value.to(device)
    for key, value in encoded.items()
}

with torch.inference_mode():
    outputs = teacher_model(**encoded)

logits = outputs.logits

assert logits.shape == (4, 4)
assert torch.isfinite(logits).all().item()

predictions = logits.argmax(dim=-1).cpu().tolist()

# ============================================================
# 5. SHOW RESULTS
# ============================================================

label_names = [
    "WORLD",
    "SPORTS",
    "BUSINESS",
    "SCI/TECH"
]

print("\n" + "-" * 65)
print("TEACHER INFERENCE CHECK")
print("-" * 65)

for i in range(4):
    print(f"\nArticle {i + 1}")
    print("Text:", sample_texts[i][:120])
    print("True class:", label_names[sample_labels[i]])
    print("Teacher prediction:", label_names[predictions[i]])

print("\n" + "=" * 65)

print("Total teacher parameters:", f"{teacher_params:,}")
print("Trainable parameters:", trainable_params)
print("Output logits shape:", tuple(logits.shape))
print("Finite logits:", True)
print("Teacher evaluation mode:", not teacher_model.training)

print("\n✅ Existing teacher checkpoint loaded successfully.")
print("✅ Teacher parameters are frozen.")
print("✅ Four-class inference verified.")
print("✅ No checkpoint files modified.")

print("=" * 65)


SEED-1598 — TEACHER MODEL VERIFICATION

Loading existing BERT teacher checkpoint...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Teacher model type: bert
Number of classes: 4
Device: cuda

-----------------------------------------------------------------
TEACHER INFERENCE CHECK
-----------------------------------------------------------------

Article 1
Text: EU go-ahead for Oracle takeover The European Commission gives US software giant Oracle the green light for its takeover 
True class: BUSINESS
Teacher prediction: BUSINESS

Article 2
Text: Embattled Blair pushes for third term Here we are again .. my toughest week yet since the last one ... and before the ne
True class: WORLD
Teacher prediction: WORLD

Article 3
Text: Coyotes asked him but Gretzky likely not in line to coach NHL club PHOENIX (CP) - Coyotes GM Mike Barnett approached man
True class: SPORTS
Teacher prediction: SPORTS

Article 4
Text: Calpers battle on corporate governance may only just be &lt;b&gt;...&lt;/b&gt; The ousting of fund president Sean Harrig
True class: BUSINESS
Teacher prediction: BUSINESS

Total teacher parameters: 109,485,316
Tr

In [ ]:

import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm

# ============================================================
# 1. CONFIGURATION
# ============================================================

assert SEED == SID4 == 1598
assert len(tokenized_train) == 108000
assert len(train_data) == 108000
assert teacher_model.config.num_labels == 4
assert not teacher_model.training

BATCH_SIZE = 64

CACHE_DIR = NEW_PART2 / "teacher_logits"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

LOGITS_PATH = CACHE_DIR / "teacher_train_logits.npy"
METADATA_PATH = CACHE_DIR / "teacher_logits_metadata.json"

SPLIT_PATH = NEW_PART2 / "data_split_manifest.json"

split_sha256 = hashlib.sha256(
    SPLIT_PATH.read_bytes()
).hexdigest()

with open(SPLIT_PATH, encoding="utf-8") as f:
    split_manifest = json.load(f)

assert split_manifest["seed"] == 1598
assert split_manifest["train_size"] == 108000

# Never write to the original seed-42 directory.
assert NEW_PART2 != OLD_PART2
assert OLD_PART2 not in LOGITS_PATH.parents

# ============================================================
# 2. PROTECT EXISTING CORRECTED CACHE
# ============================================================

if LOGITS_PATH.exists() or METADATA_PATH.exists():
    assert LOGITS_PATH.exists() and METADATA_PATH.exists(), (
        "Incomplete existing cache found. Stop and inspect."
    )

    existing_logits = np.load(LOGITS_PATH, mmap_mode="r")

    with open(METADATA_PATH, encoding="utf-8") as f:
        existing_meta = json.load(f)

    assert existing_logits.shape == (108000, 4)
    assert existing_meta["seed"] == 1598
    assert existing_meta["split_manifest_sha256"] == split_sha256
    assert existing_meta["max_length"] == 256

    print("✅ Existing seed-1598 teacher-logit cache verified.")
    print("Shape:", existing_logits.shape)
    print("Path:", LOGITS_PATH)
    print("No extraction or overwrite performed.")

else:
    # ========================================================
    # 3. EXTRACT TEACHER LOGITS
    # ========================================================

    device = next(teacher_model.parameters()).device

    teacher_model.eval()

    n_examples = len(tokenized_train)
    n_batches = math.ceil(n_examples / BATCH_SIZE)

    all_logits = np.empty(
        (n_examples, 4),
        dtype=np.float32
    )

    print("=" * 65)
    print("SEED-1598 TEACHER LOGIT EXTRACTION")
    print("=" * 65)
    print("Samples:", n_examples)
    print("Batch size:", BATCH_SIZE)
    print("Batches:", n_batches)
    print("Device:", device)
    print("Output shape:", all_logits.shape)

    for start in tqdm(
        range(0, n_examples, BATCH_SIZE),
        total=n_batches,
        desc="Extracting teacher logits"
    ):
        end = min(start + BATCH_SIZE, n_examples)

        batch_rows = tokenized_train[start:end]

        model_inputs = {
            "input_ids": batch_rows["input_ids"],
            "attention_mask": batch_rows["attention_mask"]
        }

        if "token_type_ids" in batch_rows:
            model_inputs["token_type_ids"] = (
                batch_rows["token_type_ids"]
            )

        padded = teacher_tokenizer.pad(
            model_inputs,
            padding=True,
            return_tensors="pt"
        )

        padded = {
            key: value.to(device)
            for key, value in padded.items()
        }

        with torch.inference_mode():
            logits = teacher_model(**padded).logits

        assert logits.shape == (end - start, 4)

        all_logits[start:end] = (
            logits.detach().float().cpu().numpy()
        )

    # ========================================================
    # 4. VERIFY ALL EXTRACTED RESULTS
    # ========================================================

    assert all_logits.shape == (108000, 4)
    assert np.isfinite(all_logits).all()

    predictions = np.argmax(all_logits, axis=1)

    training_labels = np.array(
        train_data["label"],
        dtype=np.int64
    )

    teacher_training_accuracy = float(
        (predictions == training_labels).mean()
    )

    # ========================================================
    # 5. SAVE LOGITS AND METADATA
    # ========================================================

    metadata = {
        "seed": SEED,
        "teacher_checkpoint": str(TEACHER_DIR),
        "dataset": "fancyzhx/ag_news",
        "split": "seed1598_training",
        "num_examples": n_examples,
        "num_classes": 4,
        "max_length": 256,
        "batch_size": BATCH_SIZE,
        "logits_dtype": "float32",
        "logits_shape": [108000, 4],
        "split_manifest_sha256": split_sha256,
        "teacher_training_accuracy": (
            teacher_training_accuracy
        ),
        "generated_utc": (
            datetime.now(timezone.utc).isoformat()
        )
    }

    # Write to temporary files first.
    temp_logits = CACHE_DIR / "teacher_train_logits.tmp.npy"
    temp_meta = CACHE_DIR / "teacher_logits_metadata.tmp.json"

    assert not temp_logits.exists()
    assert not temp_meta.exists()

    np.save(temp_logits, all_logits)

    temp_meta.write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8"
    )

    # Verify the saved array before finalizing.
    verified = np.load(temp_logits)

    assert verified.shape == (108000, 4)
    assert np.array_equal(verified, all_logits)

    assert not LOGITS_PATH.exists()
    assert not METADATA_PATH.exists()

    os.replace(temp_logits, LOGITS_PATH)
    os.replace(temp_meta, METADATA_PATH)

    print("\n" + "=" * 65)
    print("SEED-1598 TEACHER LOGITS — FINAL RESULTS")
    print("=" * 65)

    print("Logit shape:", all_logits.shape)
    print("Logit dtype:", all_logits.dtype)
    print(
        "Teacher training accuracy:",
        f"{teacher_training_accuracy * 100:.2f}%"
    )

    print("\nFirst article:")
    print("True label:", int(training_labels[0]))
    print("Teacher logits:", all_logits[0].tolist())
    print("Teacher prediction:", int(predictions[0]))

    print("\nFiles saved:")
    print(LOGITS_PATH)
    print(METADATA_PATH)

    print("\n✅ All 108,000 logits generated.")
    print("✅ All logit values are finite.")
    print("✅ Cache matches the seed-1598 split manifest.")
    print("✅ Original seed-42 cache untouched.")
    print("=" * 65)


SEED-1598 TEACHER LOGIT EXTRACTION
Samples: 108000
Batch size: 64
Batches: 1688
Device: cuda:0
Output shape: (108000, 4)


Extracting teacher logits:   0%|          | 0/1688 [00:00<?, ?it/s]


SEED-1598 TEACHER LOGITS — FINAL RESULTS
Logit shape: (108000, 4)
Logit dtype: float32
Teacher training accuracy: 98.96%

First article:
True label: 2
Teacher logits: [-0.033768512308597565, -5.753555774688721, 4.600063800811768, -0.6680991053581238]
Teacher prediction: 2

Files saved:
/content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/teacher_logits/teacher_train_logits.npy
/content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/teacher_logits/teacher_logits_metadata.json

✅ All 108,000 logits generated.
✅ All logit values are finite.
✅ Cache matches the seed-1598 split manifest.
✅ Original seed-42 cache untouched.


## 04 | Distillation loss and controlled training setup

Use α·cross-entropy + (1−α)·T²·KL divergence, where α weights the true-label term. Confirm the data loader, deterministic settings and forward/backward smoke test.

In [ ]:

import json
import hashlib
import numpy as np
import torch
import torch.nn.functional as F

# ============================================================
# 1. VERIFY SEED AND SAVED CACHE
# ============================================================

assert SID4 == SEED == 1598

cache_path = (
    NEW_PART2
    / "teacher_logits"
    / "teacher_train_logits.npy"
)

metadata_path = (
    NEW_PART2
    / "teacher_logits"
    / "teacher_logits_metadata.json"
)

split_path = NEW_PART2 / "data_split_manifest.json"

assert cache_path.is_file()
assert metadata_path.is_file()
assert split_path.is_file()

with open(metadata_path, encoding="utf-8") as f:
    metadata = json.load(f)

with open(split_path, encoding="utf-8") as f:
    split_manifest = json.load(f)

split_sha = hashlib.sha256(
    split_path.read_bytes()
).hexdigest()

assert metadata["seed"] == 1598
assert metadata["split_manifest_sha256"] == split_sha
assert split_manifest["seed"] == 1598
assert split_manifest["train_size"] == 108000

teacher_logits_all = np.load(
    cache_path,
    mmap_mode="r"
)

assert teacher_logits_all.shape == (108000, 4)
assert teacher_logits_all.dtype == np.float32
assert np.isfinite(teacher_logits_all).all()

assert len(tokenized_train) == 108000
assert len(train_data) == 108000

# ============================================================
# 2. STANDARDIZE THE LOSS DEFINITION
# ============================================================

def kd_loss_seed1598(
    student_logits,
    teacher_logits,
    labels,
    temperature=2.0,
    alpha=0.5
):
    """
    alpha = weight of HARD-LABEL cross-entropy.

    L = alpha * CE
        + (1 - alpha) * T^2 * KL
    """

    assert temperature > 0
    assert 0.0 <= alpha <= 1.0

    hard_loss = F.cross_entropy(
        student_logits.float(),
        labels.long()
    )

    teacher_probs = F.softmax(
        teacher_logits.detach().float()
        / temperature,
        dim=-1
    )

    student_log_probs = F.log_softmax(
        student_logits.float()
        / temperature,
        dim=-1
    )

    soft_loss = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean"
    ) * (temperature ** 2)

    total_loss = (
        alpha * hard_loss
        + (1.0 - alpha) * soft_loss
    )

    return total_loss, hard_loss, soft_loss

# ============================================================
# 3. TEST WITH REPRODUCIBLE SYNTHETIC DATA
# ============================================================

# Local RNG avoids changing the global training seed.
generator = torch.Generator(device="cpu")
generator.manual_seed(1598)

student_logits = torch.randn(
    4, 4,
    generator=generator,
    requires_grad=True
)

teacher_logits = torch.randn(
    4, 4,
    generator=generator
)

labels = torch.tensor([0, 1, 2, 3])

print("=" * 65)
print("SEED-1598 — CACHE AND KD LOSS VERIFICATION")
print("=" * 65)

print("Cache shape:", teacher_logits_all.shape)
print("Cache dtype:", teacher_logits_all.dtype)
print("Cache contains finite values: YES")
print("Split checksum matches: YES")

print("\nDISTILLATION LOSS TEST")

for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:

    total, hard, soft = kd_loss_seed1598(
        student_logits,
        teacher_logits,
        labels,
        temperature=2.0,
        alpha=alpha
    )

    expected = (
        alpha * hard
        + (1 - alpha) * soft
    )

    assert torch.allclose(
        total, expected,
        atol=1e-6
    )

    assert torch.isfinite(total)

    if alpha == 0.0:
        assert torch.allclose(total, soft)

    if alpha == 1.0:
        assert torch.allclose(total, hard)

    print(
        f"alpha={alpha:.2f} | "
        f"total={total.item():.6f} | "
        f"CE={hard.item():.6f} | "
        f"T²KL={soft.item():.6f}"
    )

# ============================================================
# 4. VERIFY STUDENT GRADIENTS
# ============================================================

student_logits.grad = None

total, hard, soft = kd_loss_seed1598(
    student_logits,
    teacher_logits,
    labels,
    temperature=2.0,
    alpha=0.25
)

total.backward()

assert student_logits.grad is not None
assert torch.isfinite(
    student_logits.grad
).all()

print("\nGradient verification: PASSED")
print("Teacher-logit cache: VERIFIED")
print("Alpha convention: CROSS-ENTROPY WEIGHT")

print("\n✅ All loss tests passed.")
print("✅ No student training performed.")
print("✅ Original checkpoints untouched.")

print("=" * 65)


SEED-1598 — CACHE AND KD LOSS VERIFICATION
Cache shape: (108000, 4)
Cache dtype: float32
Cache contains finite values: YES
Split checksum matches: YES

DISTILLATION LOSS TEST
alpha=0.00 | total=0.456532 | CE=1.802588 | T²KL=0.456532
alpha=0.25 | total=0.793046 | CE=1.802588 | T²KL=0.456532
alpha=0.50 | total=1.129560 | CE=1.802588 | T²KL=0.456532
alpha=0.75 | total=1.466074 | CE=1.802588 | T²KL=0.456532
alpha=1.00 | total=1.802588 | CE=1.802588 | T²KL=0.456532

Gradient verification: PASSED
Teacher-logit cache: VERIFIED
Alpha convention: CROSS-ENTROPY WEIGHT

✅ All loss tests passed.
✅ No student training performed.
✅ Original checkpoints untouched.


In [ ]:

import json
import hashlib
from pathlib import Path

# ============================================================
# 1. VERIFY CORRECTED EXPERIMENT STATE
# ============================================================

assert SID4 == SEED == 1598
assert len(tokenized_train) == 108000
assert len(tokenized_val) == 12000

assert teacher_logits_all.shape == (108000, 4)

CACHE_PATH = (
    NEW_PART2
    / "teacher_logits"
    / "teacher_train_logits.npy"
)

SPLIT_PATH = NEW_PART2 / "data_split_manifest.json"

assert CACHE_PATH.is_file()
assert SPLIT_PATH.is_file()

assert NEW_PART2 != OLD_PART2
assert NEW_PART3 != OLD_PART3

# ============================================================
# 2. DEFINE SHARED TRAINING CONFIGURATION
# ============================================================

TRAIN_CONFIG_1598 = {
    "seed": 1598,
    "model_name": "distilbert/distilbert-base-uncased",
    "num_labels": 4,
    "epochs": 3,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.06,
    "max_length": 256,
    "gradient_clip_norm": 1.0,
    "baseline_temperature": 2.0,
    "baseline_alpha": 0.5,
    "alpha_definition": "cross_entropy_weight",
    "selection_metric": "validation_accuracy",
    "teacher_logits_path": str(CACHE_PATH),
    "training_examples": 108000,
    "validation_examples": 12000,
    "split_manifest_sha256": hashlib.sha256(
        SPLIT_PATH.read_bytes()
    ).hexdigest()
}

# ============================================================
# 3. VALIDATE BEFORE SAVING
# ============================================================

assert TRAIN_CONFIG_1598["seed"] == SEED
assert TRAIN_CONFIG_1598["epochs"] == 3
assert TRAIN_CONFIG_1598["batch_size"] == 16
assert TRAIN_CONFIG_1598["num_labels"] == 4
assert TRAIN_CONFIG_1598["alpha_definition"] == (
    "cross_entropy_weight"
)

assert Path(
    TRAIN_CONFIG_1598["teacher_logits_path"]
).resolve() == CACHE_PATH.resolve()

# ============================================================
# 4. SAVE WITHOUT OVERWRITING
# ============================================================

CONFIG_PATH = NEW_PART2 / "training_config_seed1598.json"

if CONFIG_PATH.exists():
    existing = json.loads(
        CONFIG_PATH.read_text(encoding="utf-8")
    )

    assert existing == TRAIN_CONFIG_1598, (
        "Existing training configuration differs. "
        "Inspect before overwriting."
    )

    print("✅ Existing identical configuration verified.")

else:
    CONFIG_PATH.write_text(
        json.dumps(TRAIN_CONFIG_1598, indent=2),
        encoding="utf-8"
    )

    print("✅ New training configuration saved.")

# ============================================================
# 5. CHECK THE FIRST BASELINE OUTPUT PATH
# ============================================================

BASELINE_DIR = (
    NEW_PART2 / "kd_baseline_T2_alpha05"
)

assert NEW_PART2 in BASELINE_DIR.parents
assert OLD_PART2 not in BASELINE_DIR.parents

print("\n" + "=" * 65)
print("SEED-1598 — STUDENT TRAINING READINESS")
print("=" * 65)

for key, value in TRAIN_CONFIG_1598.items():
    print(f"{key:<25}: {value}")

print("\nBaseline output directory:")
print(BASELINE_DIR)

print("\nBaseline directory already exists:",
      BASELINE_DIR.exists())

print("\nConfiguration file:")
print(CONFIG_PATH)

print("\n✅ Correct seed configured.")
print("✅ Correct teacher-logit path configured.")
print("✅ Original hyperparameters preserved.")
print("✅ New output directory selected.")
print("✅ No model training performed.")

print("=" * 65)


✅ New training configuration saved.

SEED-1598 — STUDENT TRAINING READINESS
seed                     : 1598
model_name               : distilbert/distilbert-base-uncased
num_labels               : 4
epochs                   : 3
batch_size               : 16
learning_rate            : 2e-05
weight_decay             : 0.01
warmup_ratio             : 0.06
max_length               : 256
gradient_clip_norm       : 1.0
baseline_temperature     : 2.0
baseline_alpha           : 0.5
alpha_definition         : cross_entropy_weight
selection_metric         : validation_accuracy
teacher_logits_path      : /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/teacher_logits/teacher_train_logits.npy
training_examples        : 108000
validation_examples      : 12000
split_manifest_sha256    : b6cbd82e6f2e70d89cedc93b6a5cfbb36318c8d4a62b5950a4574fc2b5e5cc4a

Baseline output directory:
/content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/kd_baseline_T2_alpha05

Base

In [ ]:

import math
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader

# ============================================================
# 1. CONFIRM CORRECTED EXPERIMENT SETTINGS
# ============================================================

assert SEED == SID4 == 1598
assert TRAIN_CONFIG_1598["batch_size"] == 16
assert TRAIN_CONFIG_1598["max_length"] == 256

assert len(tokenized_train) == 108000
assert len(tokenized_val) == 12000

assert teacher_logits_all.shape == (108000, 4)

BATCH_SIZE = TRAIN_CONFIG_1598["batch_size"]

# ============================================================
# 2. DEFINE INDEXED DATASET
# ============================================================

class IndexedAGNewsDataset(Dataset):

    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        row = self.dataset[int(index)]

        return {
            "input_ids": row["input_ids"],
            "attention_mask": row["attention_mask"],
            "label": int(row["label"]),
            "row_index": int(index)
        }

train_dataset_1598 = IndexedAGNewsDataset(
    tokenized_train
)

val_dataset_1598 = IndexedAGNewsDataset(
    tokenized_val
)

# ============================================================
# 3. DEFINE BATCH COLLATOR
# ============================================================

def collate_agnews_1598(examples, include_teacher=True):

    token_inputs = [
        {
            "input_ids": item["input_ids"],
            "attention_mask": item["attention_mask"]
        }
        for item in examples
    ]

    batch = student_tokenizer.pad(
        token_inputs,
        padding=True,
        return_tensors="pt"
    )

    batch["labels"] = torch.tensor(
        [item["label"] for item in examples],
        dtype=torch.long
    )

    indices = np.asarray(
        [item["row_index"] for item in examples],
        dtype=np.int64
    )

    batch["row_indices"] = torch.from_numpy(
        indices.copy()
    )

    if include_teacher:
        matching_logits = np.asarray(
            teacher_logits_all[indices],
            dtype=np.float32
        )

        batch["teacher_logits"] = torch.from_numpy(
            matching_logits.copy()
        )

    return batch

# ============================================================
# 4. CREATE REPRODUCIBLE DATA LOADERS
# ============================================================

def make_train_loader_1598():

    # New generator for each experiment.
    # Prevents earlier batch inspection from changing
    # the actual training order.
    shuffle_generator = torch.Generator()
    shuffle_generator.manual_seed(SEED)

    return DataLoader(
        train_dataset_1598,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=shuffle_generator,
        collate_fn=lambda examples: collate_agnews_1598(
            examples,
            include_teacher=True
        ),
        num_workers=0,
        drop_last=False
    )

def make_val_loader_1598():

    return DataLoader(
        val_dataset_1598,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda examples: collate_agnews_1598(
            examples,
            include_teacher=False
        ),
        num_workers=0,
        drop_last=False
    )

# ============================================================
# 5. VERIFY ONE TRAINING BATCH
# ============================================================

train_loader_check = make_train_loader_1598()
val_loader_check = make_val_loader_1598()

train_batch = next(iter(train_loader_check))
val_batch = next(iter(val_loader_check))

assert train_batch["input_ids"].shape[0] == 16
assert train_batch["teacher_logits"].shape == (16, 4)
assert train_batch["labels"].shape == (16,)

assert val_batch["input_ids"].shape[0] == 16
assert "teacher_logits" not in val_batch

assert train_batch["input_ids"].shape[1] <= 256
assert val_batch["input_ids"].shape[1] <= 256

assert torch.isfinite(
    train_batch["teacher_logits"]
).all()

# ============================================================
# 6. VERIFY EXACT ROW-INDEX ALIGNMENT
# ============================================================

indices = train_batch["row_indices"].tolist()

for position, index in enumerate(indices):

    expected_label = int(
        tokenized_train[index]["label"]
    )

    actual_label = int(
        train_batch["labels"][position]
    )

    assert expected_label == actual_label

    expected_logits = np.asarray(
        teacher_logits_all[index]
    )

    actual_logits = (
        train_batch["teacher_logits"][position]
        .numpy()
    )

    assert np.array_equal(
        actual_logits,
        expected_logits
    )

# ============================================================
# 7. CHECK REPRODUCIBLE SHUFFLING
# ============================================================

first_order = next(
    iter(make_train_loader_1598())
)["row_indices"]

second_order = next(
    iter(make_train_loader_1598())
)["row_indices"]

assert torch.equal(
    first_order,
    second_order
)

# ============================================================
# 8. PRINT FINAL VERIFICATION
# ============================================================

print("=" * 65)
print("SEED-1598 — TRAINING BATCH VERIFICATION")
print("=" * 65)

print("Training examples:", len(train_dataset_1598))
print("Validation examples:", len(val_dataset_1598))

print("\nBatch size:", BATCH_SIZE)

print(
    "Training batches:",
    math.ceil(len(train_dataset_1598) / BATCH_SIZE)
)

print(
    "Validation batches:",
    math.ceil(len(val_dataset_1598) / BATCH_SIZE)
)

print("\nFirst training batch:")
print("Token IDs shape:", tuple(train_batch["input_ids"].shape))
print("Labels shape:", tuple(train_batch["labels"].shape))
print("Teacher logits shape:", tuple(train_batch["teacher_logits"].shape))

print("\nFirst five shuffled row indices:", indices[:5])

print("\nRow-index/logit alignment: PASSED")
print("Label alignment: PASSED")
print("Reproducible shuffling: PASSED")
print("Validation loader: PASSED")

print("\n✅ Training batches are ready.")
print("✅ No student training performed.")
print("✅ Original checkpoints untouched.")

print("=" * 65)


SEED-1598 — TRAINING BATCH VERIFICATION
Training examples: 108000
Validation examples: 12000

Batch size: 16
Training batches: 6750
Validation batches: 750

First training batch:
Token IDs shape: (16, 86)
Labels shape: (16,)
Teacher logits shape: (16, 4)

First five shuffled row indices: [36414, 79864, 3744, 74287, 90962]

Row-index/logit alignment: PASSED
Label alignment: PASSED
Reproducible shuffling: PASSED
Validation loader: PASSED

✅ Training batches are ready.
✅ No student training performed.
✅ Original checkpoints untouched.


In [ ]:

import gc
import random
import numpy as np
import torch

from transformers import (
    AutoModelForSequenceClassification,
    set_seed
)

# ============================================================
# 1. VERIFY EXPERIMENT CONFIGURATION
# ============================================================

assert SID4 == SEED == 1598
assert TRAIN_CONFIG_1598["num_labels"] == 4
assert TRAIN_CONFIG_1598["batch_size"] == 16

MODEL_NAME = TRAIN_CONFIG_1598["model_name"]

TEMPERATURE = 2.0
ALPHA = 0.5

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# ============================================================
# 2. INITIALIZE STUDENT REPRODUCIBLY
# ============================================================

set_seed(SEED)

print("=" * 65)
print("SEED-1598 — DISTILBERT SMOKE TEST")
print("=" * 65)

print("\nInitializing student...")

smoke_student = (
    AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=4
    )
)

smoke_student.to(device)
smoke_student.train()

student_parameters = sum(
    parameter.numel()
    for parameter in smoke_student.parameters()
)

print("Model:", MODEL_NAME)
print("Parameters:", f"{student_parameters:,}")
print("Device:", device)

# ============================================================
# 3. GET A VERIFIED TRAINING BATCH
# ============================================================

batch = next(
    iter(make_train_loader_1598())
)

inputs = {
    "input_ids": batch["input_ids"].to(device),
    "attention_mask": batch["attention_mask"].to(device)
}

labels = batch["labels"].to(device)

batch_teacher_logits = (
    batch["teacher_logits"].to(device)
)

assert inputs["input_ids"].shape[0] == 16
assert batch_teacher_logits.shape == (16, 4)

# ============================================================
# 4. FORWARD PASS
# ============================================================

outputs = smoke_student(**inputs)

student_logits = outputs.logits

assert student_logits.shape == (16, 4)

total_loss, ce_loss, kl_loss = kd_loss_seed1598(
    student_logits=student_logits,
    teacher_logits=batch_teacher_logits,
    labels=labels,
    temperature=TEMPERATURE,
    alpha=ALPHA
)

assert torch.isfinite(total_loss)
assert torch.isfinite(ce_loss)
assert torch.isfinite(kl_loss)

# ============================================================
# 5. BACKWARD PASS — NO OPTIMIZER UPDATE
# ============================================================

total_loss.backward()

gradient_count = 0

for parameter in smoke_student.parameters():
    if parameter.grad is not None:
        assert torch.isfinite(parameter.grad).all()
        gradient_count += 1

assert gradient_count > 0

# ============================================================
# 6. DISPLAY RESULTS
# ============================================================

print("\n" + "-" * 65)
print("TRAINING SMOKE TEST RESULTS")
print("-" * 65)

print("Seed:", SEED)
print("Temperature:", TEMPERATURE)
print("Alpha:", ALPHA)

print("\nStudent logits:", tuple(student_logits.shape))
print("Teacher logits:", tuple(batch_teacher_logits.shape))

print(f"\nCross-entropy loss: {ce_loss.item():.6f}")
print(f"Temperature-scaled KL loss: {kl_loss.item():.6f}")
print(f"Combined KD loss: {total_loss.item():.6f}")

print("\nParameters with gradients:", gradient_count)
print("Finite losses: PASSED")
print("Finite gradients: PASSED")

print("\n✅ Forward pass successful.")
print("✅ Backward pass successful.")
print("✅ No optimizer step performed.")
print("✅ No checkpoints modified.")

print("=" * 65)

# ============================================================
# 7. RELEASE TEMPORARY STUDENT
# ============================================================

del smoke_student
del outputs
del student_logits
del total_loss
del ce_loss
del kl_loss

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Temporary smoke-test model released.")


SEED-1598 — DISTILBERT SMOKE TEST

Initializing student...


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert/distilbert-base-uncased
Parameters: 66,956,548
Device: cuda

-----------------------------------------------------------------
TRAINING SMOKE TEST RESULTS
-----------------------------------------------------------------
Seed: 1598
Temperature: 2.0
Alpha: 0.5

Student logits: (16, 4)
Teacher logits: (16, 4)

Cross-entropy loss: 1.360014
Temperature-scaled KL loss: 4.732036
Combined KD loss: 3.046025

Parameters with gradients: 104
Finite losses: PASSED
Finite gradients: PASSED

✅ Forward pass successful.
✅ Backward pass successful.
✅ No optimizer step performed.
✅ No checkpoints modified.

✅ Temporary smoke-test model released.


## 05 | Baseline student: T=2, α=0.50

Train a fresh DistilBERT student for three epochs and verify its saved checkpoint and training history. This run also supplies the T=2 entry in the temperature sweep.

In [ ]:

import gc
import json
import math
import random
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed
)

# ============================================================
# 1. CONFIGURATION AND SAFETY CHECKS
# ============================================================

assert SID4 == SEED == 1598
assert len(train_dataset_1598) == 108000
assert len(val_dataset_1598) == 12000
assert teacher_logits_all.shape == (108000, 4)

CFG = TRAIN_CONFIG_1598

assert CFG["seed"] == 1598
assert CFG["epochs"] == 3
assert CFG["batch_size"] == 16
assert CFG["alpha_definition"] == "cross_entropy_weight"

T = 2.0
ALPHA = 0.5

RUN_DIR = NEW_PART2 / "kd_baseline_T2_alpha05"

assert RUN_DIR.parent.resolve() == NEW_PART2.resolve()
assert OLD_PART2.resolve() not in RUN_DIR.resolve().parents

# Prevent an accidental rerun from overwriting any
# completed or partially completed experiment.
if RUN_DIR.exists():
    assert not any(RUN_DIR.iterdir()), (
        f"Experiment directory is not empty: {RUN_DIR}\n"
        "Stop here. Do not overwrite previous training."
    )
else:
    RUN_DIR.mkdir(parents=True)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", (
    "A CUDA GPU is required for this training run."
)

# ============================================================
# 2. SET THE STUDENT-SPECIFIC RANDOM SEED
# ============================================================

set_seed(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Release the teacher model: its logits are already cached.
if "teacher_model" in globals():
    del teacher_model

gc.collect()
torch.cuda.empty_cache()

# ============================================================
# 3. INITIALIZE A FRESH STUDENT
# ============================================================

print("=" * 65)
print("SEED-1598 — BASELINE STUDENT TRAINING")
print("=" * 65)
print("Temperature:", T)
print("Alpha:", ALPHA)
print("Epochs:", CFG["epochs"])
print("Batch size:", CFG["batch_size"])
print("GPU:", torch.cuda.get_device_name(0))
print("Output:", RUN_DIR)

student = AutoModelForSequenceClassification.from_pretrained(
    CFG["model_name"],
    num_labels=4
).to(device)

student.train()

# ============================================================
# 4. CREATE TRAINING AND VALIDATION LOADERS
# ============================================================

train_loader = make_train_loader_1598()
val_loader = make_val_loader_1598()

num_epochs = CFG["epochs"]
total_steps = len(train_loader) * num_epochs

warmup_steps = math.ceil(
    total_steps * CFG["warmup_ratio"]
)

optimizer = torch.optim.AdamW(
    student.parameters(),
    lr=CFG["learning_rate"],
    weight_decay=CFG["weight_decay"]
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# ============================================================
# 5. SAVE RUN CONFIGURATION BEFORE TRAINING
# ============================================================

run_config = {
    **CFG,
    "experiment": "kd_baseline_T2_alpha05",
    "temperature": T,
    "alpha": ALPHA,
    "precision": "fp32",
    "optimizer": "AdamW",
    "scheduler": "linear_warmup_linear_decay",
    "total_optimizer_steps": total_steps,
    "warmup_steps": warmup_steps,
    "started_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

(RUN_DIR / "run_config.json").write_text(
    json.dumps(run_config, indent=2),
    encoding="utf-8"
)

# Save tokenizer once for reproducible checkpoint loading.
student_tokenizer.save_pretrained(
    str(RUN_DIR / "tokenizer")
)

# ============================================================
# 6. TRAIN AND VALIDATE EACH EPOCH
# ============================================================

history = []

best_accuracy = -1.0
best_f1 = -1.0
best_epoch = None
best_checkpoint = None

for epoch in range(1, num_epochs + 1):

    student.train()

    total_train_loss = 0.0
    total_ce_loss = 0.0
    total_kl_loss = 0.0
    total_train_examples = 0

    progress = tqdm(
        train_loader,
        desc=f"Seed1598 Baseline Epoch {epoch}/{num_epochs}"
    )

    for batch in progress:

        inputs = {
            "input_ids": batch["input_ids"].to(device),
            "attention_mask": batch["attention_mask"].to(device)
        }

        labels = batch["labels"].to(device)

        batch_teacher_logits = (
            batch["teacher_logits"].to(device)
        )

        optimizer.zero_grad(set_to_none=True)

        outputs = student(**inputs)

        loss, ce_loss, kl_loss = kd_loss_seed1598(
            student_logits=outputs.logits,
            teacher_logits=batch_teacher_logits,
            labels=labels,
            temperature=T,
            alpha=ALPHA
        )

        assert torch.isfinite(loss).item(), (
            "Non-finite training loss detected."
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            max_norm=CFG["gradient_clip_norm"],
            error_if_nonfinite=True
        )

        optimizer.step()
        scheduler.step()

        n = labels.shape[0]

        total_train_loss += loss.item() * n
        total_ce_loss += ce_loss.item() * n
        total_kl_loss += kl_loss.item() * n

        total_train_examples += n

        progress.set_postfix(
            kd_loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    # --------------------------------------------------------
    # VALIDATION: NO TEACHER LOGITS AND NO TEST DATA
    # --------------------------------------------------------

    student.eval()

    all_predictions = []
    all_labels = []

    total_val_ce = 0.0
    total_val_examples = 0

    with torch.inference_mode():

        for batch in tqdm(
            val_loader,
            desc=f"Validation Epoch {epoch}"
        ):

            inputs = {
                "input_ids": batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device)
            }

            labels = batch["labels"].to(device)

            outputs = student(
                **inputs,
                labels=labels
            )

            n = labels.shape[0]

            total_val_ce += outputs.loss.item() * n
            total_val_examples += n

            predictions = outputs.logits.argmax(
                dim=-1
            )

            all_predictions.extend(
                predictions.cpu().tolist()
            )

            all_labels.extend(
                labels.cpu().tolist()
            )

    assert len(all_labels) == 12000
    assert len(all_predictions) == 12000

    val_accuracy = float(
        np.mean(
            np.asarray(all_predictions)
            == np.asarray(all_labels)
        )
    )

    val_macro_f1 = float(
        f1_score(
            all_labels,
            all_predictions,
            average="macro",
            labels=[0, 1, 2, 3]
        )
    )

    epoch_result = {
        "epoch": epoch,
        "train_kd_loss": (
            total_train_loss / total_train_examples
        ),
        "train_ce_loss": (
            total_ce_loss / total_train_examples
        ),
        "train_t2_kl_loss": (
            total_kl_loss / total_train_examples
        ),
        "validation_ce_loss": (
            total_val_ce / total_val_examples
        ),
        "validation_accuracy": val_accuracy,
        "validation_macro_f1": val_macro_f1,
        "optimizer_steps_completed": (
            epoch * len(train_loader)
        ),
        "completed_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    history.append(epoch_result)

    # --------------------------------------------------------
    # SAVE AN INDEPENDENT CHECKPOINT FOR EVERY EPOCH
    # --------------------------------------------------------

    checkpoint_dir = (
        RUN_DIR / f"checkpoint_epoch_{epoch}"
    )

    student.save_pretrained(
        str(checkpoint_dir),
        safe_serialization=True
    )

    # Persist history after each completed epoch.
    (RUN_DIR / "training_history.json").write_text(
        json.dumps(history, indent=2),
        encoding="utf-8"
    )

    # --------------------------------------------------------
    # SELECT BEST EPOCH USING VALIDATION ONLY
    # --------------------------------------------------------

    is_better = (
        val_accuracy > best_accuracy
        or (
            math.isclose(
                val_accuracy,
                best_accuracy,
                abs_tol=1e-12
            )
            and val_macro_f1 > best_f1
        )
    )

    if is_better:
        best_accuracy = val_accuracy
        best_f1 = val_macro_f1
        best_epoch = epoch
        best_checkpoint = checkpoint_dir

    best_so_far = {
        "best_epoch": best_epoch,
        "best_checkpoint": str(best_checkpoint),
        "validation_accuracy": best_accuracy,
        "validation_macro_f1": best_f1
    }

    (RUN_DIR / "best_checkpoint.json").write_text(
        json.dumps(best_so_far, indent=2),
        encoding="utf-8"
    )

    print("\n" + "-" * 65)
    print(f"EPOCH {epoch} RESULTS")
    print("-" * 65)

    for key, value in epoch_result.items():
        if isinstance(value, float):
            print(f"{key}: {value:.6f}")
        else:
            print(f"{key}: {value}")

    print("Checkpoint saved:", checkpoint_dir)
    print("Best epoch so far:", best_epoch)

# ============================================================
# 7. SAVE FINAL EXPERIMENT SUMMARY
# ============================================================

summary = {
    "seed": SEED,
    "experiment": "kd_baseline_T2_alpha05",
    "temperature": T,
    "alpha": ALPHA,
    "epochs_completed": len(history),
    "best_epoch": best_epoch,
    "best_checkpoint": str(best_checkpoint),
    "best_validation_accuracy": best_accuracy,
    "best_validation_macro_f1": best_f1,
    "history": history,
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat()
}

assert len(history) == 3

(RUN_DIR / "baseline_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\n" + "=" * 65)
print("SEED-1598 — BASELINE TRAINING COMPLETE")
print("=" * 65)

print("Epochs completed:", len(history))
print("Best epoch:", best_epoch)
print(
    "Best validation accuracy:",
    f"{best_accuracy * 100:.2f}%"
)
print(
    "Best validation macro-F1:",
    f"{best_f1:.4f}"
)

print("\nBest checkpoint:", best_checkpoint)
print("Experiment folder:", RUN_DIR)

print("\n✅ Baseline trained with seed 1598.")
print("✅ All epoch checkpoints saved.")
print("✅ Best checkpoint selected using validation only.")
print("✅ Original seed-42 outputs untouched.")
print("=" * 65)

# ============================================================
# 8. RELEASE GPU MEMORY
# ============================================================

del student
del optimizer
del scheduler

gc.collect()
torch.cuda.empty_cache()

print("\n✅ Baseline training model released from GPU.")


SEED-1598 — BASELINE STUDENT TRAINING
Temperature: 2.0
Alpha: 0.5
Epochs: 3
Batch size: 16
GPU: NVIDIA L4
Output: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/kd_baseline_T2_alpha05


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Seed1598 Baseline Epoch 1/3:   0%|          | 0/6750 [00:00<?, ?it/s]

Validation Epoch 1:   0%|          | 0/750 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

Seed1598 Baseline Epoch 3/3:   0%|          | 0/6750 [00:00<?, ?it/s]

Validation Epoch 3:   0%|          | 0/750 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


-----------------------------------------------------------------
EPOCH 3 RESULTS
-----------------------------------------------------------------
epoch: 3
train_kd_loss: 0.140638
train_ce_loss: 0.118948
train_t2_kl_loss: 0.162328
validation_ce_loss: 0.228407
validation_accuracy: 0.947000
validation_macro_f1: 0.947056
optimizer_steps_completed: 20250
completed_utc: 2026-11-20T00:07:38.582213+00:00
Checkpoint saved: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/kd_baseline_T2_alpha05/checkpoint_epoch_3
Best epoch so far: 3

SEED-1598 — BASELINE TRAINING COMPLETE
Epochs completed: 3
Best epoch: 3
Best validation accuracy: 94.70%
Best validation macro-F1: 0.9471

Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/kd_baseline_T2_alpha05/checkpoint_epoch_3
Experiment folder: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/kd_baseline_T2_alpha05

✅ Baseline trained with seed 1598.
✅ All epoch checkpoint

In [ ]:

import json
from pathlib import Path
import torch
from transformers import AutoModelForSequenceClassification

# ============================================================
# 1. VERIFY CORRECTED EXPERIMENT
# ============================================================

assert SEED == SID4 == 1598

BASELINE_DIR = (
    NEW_PART2 / "kd_baseline_T2_alpha05"
)

HISTORY_PATH = BASELINE_DIR / "training_history.json"
SUMMARY_PATH = BASELINE_DIR / "baseline_summary.json"
BEST_PATH = BASELINE_DIR / "best_checkpoint.json"

for path in [HISTORY_PATH, SUMMARY_PATH, BEST_PATH]:
    assert path.is_file(), f"Missing required file: {path}"

history = json.loads(HISTORY_PATH.read_text())
summary = json.loads(SUMMARY_PATH.read_text())
best = json.loads(BEST_PATH.read_text())

assert len(history) == 3
assert summary["seed"] == 1598
assert summary["temperature"] == 2.0
assert summary["alpha"] == 0.5
assert summary["epochs_completed"] == 3

assert best["best_epoch"] == 3

# ============================================================
# 2. VERIFY EVERY BASELINE CHECKPOINT
# ============================================================

for epoch in [1, 2, 3]:
    checkpoint_dir = (
        BASELINE_DIR / f"checkpoint_epoch_{epoch}"
    )

    assert (checkpoint_dir / "config.json").is_file()

    assert (
        (checkpoint_dir / "model.safetensors").is_file()
        or (checkpoint_dir / "pytorch_model.bin").is_file()
    )

# ============================================================
# 3. LOAD BEST CHECKPOINT FOR VERIFICATION ONLY
# ============================================================

best_checkpoint = Path(best["best_checkpoint"])

assert best_checkpoint.resolve() == (
    BASELINE_DIR / "checkpoint_epoch_3"
).resolve()

model = AutoModelForSequenceClassification.from_pretrained(
    str(best_checkpoint),
    local_files_only=True
)

assert model.config.num_labels == 4

model_parameters = sum(
    p.numel() for p in model.parameters()
)

assert model_parameters == 66956548

del model

# ============================================================
# 4. DEFINE REMAINING TEMPERATURE EXPERIMENTS
# ============================================================

TEMPERATURE_SWEEP_1598 = {
    1: NEW_PART2 / "temperature_T1_alpha05",
    2: BASELINE_DIR,
    5: NEW_PART2 / "temperature_T5_alpha05",
    10: NEW_PART2 / "temperature_T10_alpha05"
}

assert len(
    set(map(str, TEMPERATURE_SWEEP_1598.values()))
) == 4

for T, path in TEMPERATURE_SWEEP_1598.items():
    assert path.resolve() != OLD_PART2.resolve()
    assert NEW_PART2.resolve() in path.resolve().parents

    if T != 2:
        assert (
            not path.exists()
            or not any(path.iterdir())
        ), (
            f"Temperature T={T} already has files. "
            "Stop before overwriting."
        )

# ============================================================
# 5. PRINT FINAL READINESS SUMMARY
# ============================================================

print("=" * 65)
print("SEED-1598 — TEMPERATURE SWEEP READINESS")
print("=" * 65)

print("Baseline seed:", summary["seed"])
print("Baseline temperature:", summary["temperature"])
print("Baseline alpha:", summary["alpha"])
print("Epochs completed:", summary["epochs_completed"])

print(
    "Best validation accuracy:",
    f"{summary['best_validation_accuracy']*100:.2f}%"
)

print(
    "Best validation macro-F1:",
    f"{summary['best_validation_macro_f1']:.4f}"
)

print("Best checkpoint:", best_checkpoint)
print("Student parameters:", f"{model_parameters:,}")

print("\nTEMPERATURE EXPERIMENT STATUS")

for T, path in TEMPERATURE_SWEEP_1598.items():
    status = (
        "COMPLETE — reuse baseline"
        if T == 2
        else "READY — new training"
    )

    print(f"T={T:<2} | {status}")
    print("     ", path)

print("\n✅ Baseline checkpoints verified.")
print("✅ Saved training history verified.")
print("✅ Best student checkpoint loads correctly.")
print("✅ No duplicate T=2 training required.")
print("✅ New experiment paths protected.")
print("✅ No model training performed.")

print("=" * 65)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

SEED-1598 — TEMPERATURE SWEEP READINESS
Baseline seed: 1598
Baseline temperature: 2.0
Baseline alpha: 0.5
Epochs completed: 3
Best validation accuracy: 94.70%
Best validation macro-F1: 0.9471
Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/kd_baseline_T2_alpha05/checkpoint_epoch_3
Student parameters: 66,956,548

TEMPERATURE EXPERIMENT STATUS
T=1  | READY — new training
      /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T1_alpha05
T=2  | COMPLETE — reuse baseline
      /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/kd_baseline_T2_alpha05
T=5  | READY — new training
      /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T5_alpha05
T=10 | READY — new training
      /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T10_alpha05

✅ Baseline checkpoints verified.
✅ Saved training history verified.
✅ Best student checkpoin

## 06 | Temperature sweep: T ∈ {1,2,5,10}

Hold α=0.50 and all other training settings fixed, reinitialize each student, and select T using validation accuracy only. The best temperature is T=5.

In [ ]:

import gc
import json
import math
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from transformers import (
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed
)

assert SEED == SID4 == 1598
assert teacher_logits_all.shape == (108000, 4)

def train_kd_experiment_1598(
    temperature,
    alpha,
    run_dir
):
    """
    Train one independent seed-1598 DistilBERT experiment.

    Uses:
      alpha * CE + (1-alpha) * T^2 * KL

    Does not overwrite any existing experiment.
    """

    assert temperature > 0
    assert 0 <= alpha <= 1

    run_dir = Path(run_dir)

    # ========================================================
    # 1. PROTECT ORIGINAL AND EXISTING EXPERIMENTS
    # ========================================================

    assert run_dir.resolve().parent == NEW_PART2.resolve()
    assert run_dir.resolve() != BASELINE_DIR.resolve()
    assert OLD_PART2.resolve() not in run_dir.resolve().parents

    if run_dir.exists():
        assert not any(run_dir.iterdir()), (
            f"Experiment directory already contains files: "
            f"{run_dir}. Stop to prevent overwriting."
        )
    else:
        run_dir.mkdir(parents=True)

    # ========================================================
    # 2. RESET SEED BEFORE EVERY INDEPENDENT RUN
    # ========================================================

    set_seed(1598)

    device = torch.device("cuda:0")
    assert torch.cuda.is_available()

    model = AutoModelForSequenceClassification.from_pretrained(
        TRAIN_CONFIG_1598["model_name"],
        num_labels=4
    ).to(device)

    train_loader = make_train_loader_1598()
    val_loader = make_val_loader_1598()

    epochs = TRAIN_CONFIG_1598["epochs"]
    total_steps = len(train_loader) * epochs

    warmup_steps = math.ceil(
        total_steps * TRAIN_CONFIG_1598["warmup_ratio"]
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_CONFIG_1598["learning_rate"],
        weight_decay=TRAIN_CONFIG_1598["weight_decay"]
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # ========================================================
    # 3. SAVE THE EXACT RUN CONFIGURATION
    # ========================================================

    run_config = {
        **TRAIN_CONFIG_1598,
        "experiment": run_dir.name,
        "temperature": float(temperature),
        "alpha": float(alpha),
        "precision": "fp32",
        "optimizer": "AdamW",
        "scheduler": "linear_warmup_linear_decay",
        "total_optimizer_steps": total_steps,
        "warmup_steps": warmup_steps,
        "started_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    (run_dir / "run_config.json").write_text(
        json.dumps(run_config, indent=2),
        encoding="utf-8"
    )

    student_tokenizer.save_pretrained(
        str(run_dir / "tokenizer")
    )

    # ========================================================
    # 4. TRAIN FOR THREE EPOCHS
    # ========================================================

    history = []
    best_accuracy = -1.0
    best_f1 = -1.0
    best_epoch = None
    best_checkpoint = None

    print("\n" + "=" * 65)
    print("SEED-1598 — KNOWLEDGE DISTILLATION")
    print("=" * 65)
    print("Temperature:", temperature)
    print("Alpha:", alpha)
    print("Seed:", SEED)
    print("Epochs:", epochs)
    print("GPU:", torch.cuda.get_device_name(0))
    print("Output:", run_dir)

    for epoch in range(1, epochs + 1):

        model.train()

        train_loss_sum = 0.0
        train_ce_sum = 0.0
        train_kl_sum = 0.0
        train_examples = 0

        for batch in tqdm(
            train_loader,
            desc=f"T={temperature} Epoch {epoch}/{epochs}"
        ):

            inputs = {
                "input_ids": batch["input_ids"].to(device),
                "attention_mask": batch["attention_mask"].to(device)
            }

            labels = batch["labels"].to(device)

            teacher_logits = (
                batch["teacher_logits"].to(device)
            )

            optimizer.zero_grad(set_to_none=True)

            outputs = model(**inputs)

            loss, ce_loss, kl_loss = kd_loss_seed1598(
                student_logits=outputs.logits,
                teacher_logits=teacher_logits,
                labels=labels,
                temperature=temperature,
                alpha=alpha
            )

            assert torch.isfinite(loss).item(), (
                "Non-finite training loss detected."
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                TRAIN_CONFIG_1598["gradient_clip_norm"],
                error_if_nonfinite=True
            )

            optimizer.step()
            scheduler.step()

            n = labels.size(0)

            train_loss_sum += loss.item() * n
            train_ce_sum += ce_loss.item() * n
            train_kl_sum += kl_loss.item() * n
            train_examples += n

        # ====================================================
        # 5. VALIDATE AFTER EACH EPOCH
        # ====================================================

        model.eval()

        predictions = []
        true_labels = []

        val_ce_sum = 0.0
        val_examples = 0

        with torch.inference_mode():

            for batch in tqdm(
                val_loader,
                desc=f"T={temperature} Validation {epoch}"
            ):

                inputs = {
                    "input_ids": batch["input_ids"].to(device),
                    "attention_mask": batch["attention_mask"].to(device)
                }

                labels = batch["labels"].to(device)

                outputs = model(
                    **inputs,
                    labels=labels
                )

                n = labels.size(0)

                val_ce_sum += outputs.loss.item() * n
                val_examples += n

                predicted = outputs.logits.argmax(
                    dim=-1
                )

                predictions.extend(
                    predicted.cpu().tolist()
                )

                true_labels.extend(
                    labels.cpu().tolist()
                )

        assert len(predictions) == 12000
        assert len(true_labels) == 12000

        accuracy = float(
            np.mean(
                np.array(predictions)
                == np.array(true_labels)
            )
        )

        macro_f1 = float(
            f1_score(
                true_labels,
                predictions,
                labels=[0, 1, 2, 3],
                average="macro"
            )
        )

        epoch_metrics = {
            "epoch": epoch,
            "train_kd_loss": train_loss_sum / train_examples,
            "train_ce_loss": train_ce_sum / train_examples,
            "train_t2_kl_loss": train_kl_sum / train_examples,
            "validation_ce_loss": val_ce_sum / val_examples,
            "validation_accuracy": accuracy,
            "validation_macro_f1": macro_f1,
            "optimizer_steps_completed": (
                epoch * len(train_loader)
            ),
            "completed_utc": datetime.now(
                timezone.utc
            ).isoformat()
        }

        history.append(epoch_metrics)

        # Save a separate checkpoint for each epoch.
        checkpoint_dir = (
            run_dir / f"checkpoint_epoch_{epoch}"
        )

        model.save_pretrained(
            str(checkpoint_dir),
            safe_serialization=True
        )

        (run_dir / "training_history.json").write_text(
            json.dumps(history, indent=2),
            encoding="utf-8"
        )

        # Select using validation only.
        better = (
            accuracy > best_accuracy
            or (
                math.isclose(
                    accuracy,
                    best_accuracy,
                    abs_tol=1e-12
                )
                and macro_f1 > best_f1
            )
        )

        if better:
            best_accuracy = accuracy
            best_f1 = macro_f1
            best_epoch = epoch
            best_checkpoint = checkpoint_dir

        best_info = {
            "best_epoch": best_epoch,
            "best_checkpoint": str(best_checkpoint),
            "validation_accuracy": best_accuracy,
            "validation_macro_f1": best_f1
        }

        (run_dir / "best_checkpoint.json").write_text(
            json.dumps(best_info, indent=2),
            encoding="utf-8"
        )

        print("\n" + "-" * 65)
        print(f"T={temperature}, alpha={alpha}, Epoch {epoch}")
        print("-" * 65)
        print(
            "Train KD loss:",
            f"{epoch_metrics['train_kd_loss']:.6f}"
        )
        print(
            "Validation accuracy:",
            f"{accuracy * 100:.2f}%"
        )
        print(
            "Validation macro-F1:",
            f"{macro_f1:.4f}"
        )
        print("Best epoch so far:", best_epoch)

    # ========================================================
    # 6. SAVE FINAL SUMMARY
    # ========================================================

    assert len(history) == epochs

    summary = {
        "seed": SEED,
        "experiment": run_dir.name,
        "temperature": float(temperature),
        "alpha": float(alpha),
        "epochs_completed": epochs,
        "best_epoch": best_epoch,
        "best_checkpoint": str(best_checkpoint),
        "best_validation_accuracy": best_accuracy,
        "best_validation_macro_f1": best_f1,
        "history": history,
        "completed_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    (run_dir / "experiment_summary.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8"
    )

    print("\n" + "=" * 65)
    print("EXPERIMENT COMPLETE")
    print("=" * 65)
    print("Temperature:", temperature)
    print("Alpha:", alpha)
    print("Seed:", SEED)
    print("Best epoch:", best_epoch)
    print(
        "Best validation accuracy:",
        f"{best_accuracy * 100:.2f}%"
    )
    print(
        "Best validation macro-F1:",
        f"{best_f1:.4f}"
    )
    print("Best checkpoint:", best_checkpoint)
    print("Experiment folder:", run_dir)

    del model
    del optimizer
    del scheduler

    gc.collect()
    torch.cuda.empty_cache()

    return summary

print("=" * 65)
print("SEED-1598 — REUSABLE KD TRAINING FUNCTION")
print("=" * 65)
print("✅ Training function defined.")
print("✅ Same seed and hyperparameters for every run.")
print("✅ Baseline T=2 will be reused.")
print("✅ Existing experiment directories protected.")
print("✅ No training performed yet.")
print("=" * 65)


SEED-1598 — REUSABLE KD TRAINING FUNCTION
✅ Training function defined.
✅ Same seed and hyperparameters for every run.
✅ Baseline T=2 will be reused.
✅ Existing experiment directories protected.
✅ No training performed yet.


In [ ]:

# First new temperature experiment: T=1, alpha=0.5

result_T1 = train_kd_experiment_1598(
    temperature=1.0,
    alpha=0.5,
    run_dir=TEMPERATURE_SWEEP_1598[1]
)

print("\nT=1 TRAINING FINISHED")
print(
    "Best validation accuracy:",
    f"{result_T1['best_validation_accuracy'] * 100:.2f}%"
)
print(
    "Best validation macro-F1:",
    f"{result_T1['best_validation_macro_f1']:.4f}"
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



SEED-1598 — KNOWLEDGE DISTILLATION
Temperature: 1.0
Alpha: 0.5
Seed: 1598
Epochs: 3
GPU: NVIDIA L4
Output: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T1_alpha05


T=1.0 Epoch 1/3:   0%|          | 0/6750 [00:00<?, ?it/s]

T=1.0 Validation 1:   0%|          | 0/750 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

T=1.0 Epoch 3/3:   0%|          | 0/6750 [00:00<?, ?it/s]

T=1.0 Validation 3:   0%|          | 0/750 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


-----------------------------------------------------------------
T=1.0, alpha=0.5, Epoch 3
-----------------------------------------------------------------
Train KD loss: 0.080579
Validation accuracy: 94.72%
Validation macro-F1: 0.9472
Best epoch so far: 3

EXPERIMENT COMPLETE
Temperature: 1.0
Alpha: 0.5
Seed: 1598
Best epoch: 3
Best validation accuracy: 94.72%
Best validation macro-F1: 0.9472
Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T1_alpha05/checkpoint_epoch_3
Experiment folder: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T1_alpha05

T=1 TRAINING FINISHED
Best validation accuracy: 94.72%
Best validation macro-F1: 0.9472


In [ ]:

# ============================================================
# STEP 16 — TEMPERATURE T=5 EXPERIMENT
# ============================================================

assert SEED == SID4 == 1598

T5_DIR = TEMPERATURE_SWEEP_1598[5]

assert T5_DIR == (
    NEW_PART2 / "temperature_T5_alpha05"
)

# Protect existing results.
assert (
    not T5_DIR.exists()
    or not any(T5_DIR.iterdir())
), "T=5 output directory is not empty. Do not overwrite it."

result_T5 = train_kd_experiment_1598(
    temperature=5.0,
    alpha=0.5,
    run_dir=T5_DIR
)

assert result_T5["seed"] == 1598
assert result_T5["temperature"] == 5.0
assert result_T5["alpha"] == 0.5
assert result_T5["epochs_completed"] == 3

print("\n" + "=" * 65)
print("SEED-1598 — T=5 FINAL RESULTS")
print("=" * 65)

print("Best epoch:", result_T5["best_epoch"])

print(
    "Best validation accuracy:",
    f"{result_T5['best_validation_accuracy'] * 100:.2f}%"
)

print(
    "Best validation macro-F1:",
    f"{result_T5['best_validation_macro_f1']:.4f}"
)

print("Best checkpoint:", result_T5["best_checkpoint"])

print("\n✅ T=5 training completed.")
print("✅ All three epoch checkpoints saved.")
print("✅ Results saved separately.")
print("✅ Original experiments untouched.")

print("=" * 65)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



SEED-1598 — KNOWLEDGE DISTILLATION
Temperature: 5.0
Alpha: 0.5
Seed: 1598
Epochs: 3
GPU: NVIDIA L4
Output: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T5_alpha05


T=5.0 Epoch 1/3:   0%|          | 0/6750 [00:00<?, ?it/s]

T=5.0 Validation 1:   0%|          | 0/750 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

T=5.0 Epoch 3/3:   0%|          | 0/6750 [00:00<?, ?it/s]

T=5.0 Validation 3:   0%|          | 0/750 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


-----------------------------------------------------------------
T=5.0, alpha=0.5, Epoch 3
-----------------------------------------------------------------
Train KD loss: 0.191567
Validation accuracy: 94.77%
Validation macro-F1: 0.9478
Best epoch so far: 3

EXPERIMENT COMPLETE
Temperature: 5.0
Alpha: 0.5
Seed: 1598
Best epoch: 3
Best validation accuracy: 94.77%
Best validation macro-F1: 0.9478
Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T5_alpha05/checkpoint_epoch_3
Experiment folder: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T5_alpha05

SEED-1598 — T=5 FINAL RESULTS
Best epoch: 3
Best validation accuracy: 94.77%
Best validation macro-F1: 0.9478
Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T5_alpha05/checkpoint_epoch_3

✅ T=5 training completed.
✅ All three epoch checkpoints saved.
✅ Results saved separately.
✅ Original experiment

In [ ]:

# ============================================================
# STEP 17 — FINAL TEMPERATURE EXPERIMENT: T=10
# ============================================================

from pathlib import Path

assert SEED == SID4 == 1598

T10_DIR = TEMPERATURE_SWEEP_1598[10]

assert T10_DIR.resolve() == (
    NEW_PART2 / "temperature_T10_alpha05"
).resolve()

# Prevent any existing experiment from being overwritten.
assert (
    not T10_DIR.exists()
    or not any(T10_DIR.iterdir())
), (
    "T=10 directory already contains files. "
    "Stop and inspect before rerunning."
)

# Train the final temperature configuration.
result_T10 = train_kd_experiment_1598(
    temperature=10.0,
    alpha=0.5,
    run_dir=T10_DIR
)

# Verify the experiment completed correctly.
assert result_T10["seed"] == 1598
assert result_T10["temperature"] == 10.0
assert result_T10["alpha"] == 0.5
assert result_T10["epochs_completed"] == 3

assert Path(
    result_T10["best_checkpoint"]
).is_dir()

print("\n" + "=" * 65)
print("SEED-1598 — T=10 FINAL RESULTS")
print("=" * 65)

print("Best epoch:", result_T10["best_epoch"])

print(
    "Best validation accuracy:",
    f"{result_T10['best_validation_accuracy'] * 100:.2f}%"
)

print(
    "Best validation macro-F1:",
    f"{result_T10['best_validation_macro_f1']:.4f}"
)

print(
    "Best checkpoint:",
    result_T10["best_checkpoint"]
)

print("\n✅ T=10 training completed.")
print("✅ All three epoch checkpoints saved.")
print("✅ Temperature sweep training completed.")
print("✅ Original experiments remain untouched.")

print("=" * 65)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



SEED-1598 — KNOWLEDGE DISTILLATION
Temperature: 10.0
Alpha: 0.5
Seed: 1598
Epochs: 3
GPU: NVIDIA L4
Output: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T10_alpha05


T=10.0 Epoch 1/3:   0%|          | 0/6750 [00:00<?, ?it/s]

T=10.0 Validation 1:   0%|          | 0/750 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (5 outputs left out)

T=10.0 Epoch 3/3:   0%|          | 0/6750 [00:00<?, ?it/s]

T=10.0 Validation 3:   0%|          | 0/750 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


-----------------------------------------------------------------
T=10.0, alpha=0.5, Epoch 3
-----------------------------------------------------------------
Train KD loss: 0.188229
Validation accuracy: 94.71%
Validation macro-F1: 0.9471
Best epoch so far: 3

EXPERIMENT COMPLETE
Temperature: 10.0
Alpha: 0.5
Seed: 1598
Best epoch: 3
Best validation accuracy: 94.71%
Best validation macro-F1: 0.9471
Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T10_alpha05/checkpoint_epoch_3
Experiment folder: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T10_alpha05

SEED-1598 — T=10 FINAL RESULTS
Best epoch: 3
Best validation accuracy: 94.71%
Best validation macro-F1: 0.9471
Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T10_alpha05/checkpoint_epoch_3

✅ T=10 training completed.
✅ All three epoch checkpoints saved.
✅ Temperature sweep training completed.
✅ 

In [ ]:

import json
import math
from pathlib import Path

import pandas as pd

# ============================================================
# 1. VERIFY EXPERIMENT SETTINGS
# ============================================================

assert SEED == SID4 == 1598

TEMPERATURE_FILES = {
    1: (
        NEW_PART2 / "temperature_T1_alpha05"
        / "experiment_summary.json"
    ),
    2: (
        NEW_PART2 / "kd_baseline_T2_alpha05"
        / "baseline_summary.json"
    ),
    5: (
        NEW_PART2 / "temperature_T5_alpha05"
        / "experiment_summary.json"
    ),
    10: (
        NEW_PART2 / "temperature_T10_alpha05"
        / "experiment_summary.json"
    )
}

rows = []

# ============================================================
# 2. LOAD ACTUAL SAVED EXPERIMENT RESULTS
# ============================================================

for temperature, summary_path in TEMPERATURE_FILES.items():

    assert summary_path.is_file(), (
        f"Missing experiment summary: {summary_path}"
    )

    summary = json.loads(
        summary_path.read_text(encoding="utf-8")
    )

    assert summary["seed"] == 1598
    assert math.isclose(
        summary["temperature"],
        float(temperature)
    )
    assert math.isclose(summary["alpha"], 0.5)
    assert summary["epochs_completed"] == 3
    assert len(summary["history"]) == 3

    checkpoint = Path(summary["best_checkpoint"])

    assert checkpoint.is_dir()
    assert (
        checkpoint / "config.json"
    ).is_file()

    rows.append({
        "temperature": temperature,
        "alpha": 0.5,
        "seed": 1598,
        "best_epoch": summary["best_epoch"],
        "validation_accuracy":
            summary["best_validation_accuracy"],
        "validation_macro_f1":
            summary["best_validation_macro_f1"],
        "best_checkpoint": str(checkpoint)
    })

# ============================================================
# 3. SELECT BEST TEMPERATURE
# ============================================================

results_df = pd.DataFrame(rows)

results_df = results_df.sort_values(
    "temperature"
).reset_index(drop=True)

ranked = results_df.sort_values(
    ["validation_accuracy", "validation_macro_f1"],
    ascending=[False, False]
)

winner = ranked.iloc[0]

BEST_T_1598 = int(winner["temperature"])
BEST_T_CHECKPOINT_1598 = str(
    winner["best_checkpoint"]
)

assert BEST_T_1598 in [1, 2, 5, 10]

# ============================================================
# 4. SAVE COMPARISON AND SELECTION
# ============================================================

comparison_path = (
    NEW_PART2 / "temperature_sweep_comparison.csv"
)

selection_path = (
    NEW_PART2 / "selected_temperature.json"
)

selection = {
    "seed": 1598,
    "selection_dataset": "AG News validation",
    "selection_metric": "validation_accuracy",
    "tie_breaker": "validation_macro_f1",
    "selected_temperature": BEST_T_1598,
    "alpha": 0.5,
    "best_epoch": int(winner["best_epoch"]),
    "validation_accuracy": float(
        winner["validation_accuracy"]
    ),
    "validation_macro_f1": float(
        winner["validation_macro_f1"]
    ),
    "best_checkpoint": BEST_T_CHECKPOINT_1598
}

if comparison_path.exists():
    existing_df = pd.read_csv(comparison_path)
    pd.testing.assert_frame_equal(
        existing_df,
        results_df.drop(
            columns=["best_checkpoint"]
        ),
        check_dtype=False
    )
else:
    results_df.drop(
        columns=["best_checkpoint"]
    ).to_csv(comparison_path, index=False)

if selection_path.exists():
    previous_selection = json.loads(
        selection_path.read_text(encoding="utf-8")
    )

    assert previous_selection == selection, (
        "Existing temperature selection differs. "
        "Inspect before overwriting."
    )
else:
    selection_path.write_text(
        json.dumps(selection, indent=2),
        encoding="utf-8"
    )

# ============================================================
# 5. DISPLAY FINAL RESULTS
# ============================================================

print("=" * 65)
print("SEED-1598 — FINAL TEMPERATURE SWEEP COMPARISON")
print("=" * 65)

display(
    results_df[
        [
            "temperature",
            "alpha",
            "best_epoch",
            "validation_accuracy",
            "validation_macro_f1"
        ]
    ]
)

print("\nSELECTED TEMPERATURE:", BEST_T_1598)

print(
    "Best validation accuracy:",
    f"{selection['validation_accuracy'] * 100:.4f}%"
)

print(
    "Best validation macro-F1:",
    f"{selection['validation_macro_f1']:.6f}"
)

print(
    "Best checkpoint:",
    BEST_T_CHECKPOINT_1598
)

print("\nSaved comparison:", comparison_path)
print("Saved selection:", selection_path)

print("\n✅ All four temperatures verified.")
print("✅ Winner selected using validation only.")
print("✅ No duplicate training performed.")
print("✅ Test dataset remains unused for selection.")

print("=" * 65)


SEED-1598 — FINAL TEMPERATURE SWEEP COMPARISON


,temperature,alpha,best_epoch,validation_accuracy,validation_macro_f1
0,1,0.5,3,0.947167,0.947245
1,2,0.5,3,0.947000,0.947056
2,5,0.5,3,0.947750,0.947808
3,10,0.5,3,0.947083,0.947146



SELECTED TEMPERATURE: 5
Best validation accuracy: 94.7750%
Best validation macro-F1: 0.947808
Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_T5_alpha05/checkpoint_epoch_3

Saved comparison: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/temperature_sweep_comparison.csv
Saved selection: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/selected_temperature.json

✅ All four temperatures verified.
✅ Winner selected using validation only.
✅ No duplicate training performed.
✅ Test dataset remains unused for selection.


**Temperature sweep (validation only, α=0.50):** T=1 → 94.7167%; T=2 → 94.7000%; **T=5 → 94.7750%**; T=10 → 94.7083%. Epochs=3 per run, SEED=1598.

## 07 | Loss-weight sweep: α ∈ {0.25,0.50,0.75}

At T=5, compare three fresh-student configurations. The α=0.50 run is the previously trained T=5 experiment. Select α=0.25 using validation accuracy; checkpoint epoch 3.

In [ ]:

from pathlib import Path
import json

# ============================================================
# STEP 19 — ALPHA 0.25 EXPERIMENT
# ============================================================

assert SID4 == SEED == 1598

# Read the selected temperature from the saved evidence.
selection_path = NEW_PART2 / "selected_temperature.json"

assert selection_path.is_file()

selection = json.loads(
    selection_path.read_text(encoding="utf-8")
)

SELECTED_T = float(selection["selected_temperature"])

assert SELECTED_T == 5.0
assert selection["seed"] == 1598

# ============================================================
# SAFE NEW OUTPUT DIRECTORY
# ============================================================

ALPHA_025_DIR = NEW_PART2 / "alpha_T5_alpha025"

assert ALPHA_025_DIR.resolve().parent == NEW_PART2.resolve()

assert (
    not ALPHA_025_DIR.exists()
    or not any(ALPHA_025_DIR.iterdir())
), (
    "Alpha=0.25 experiment directory is not empty. "
    "Stop and inspect before rerunning."
)

print("=" * 65)
print("SEED-1598 — ALPHA SWEEP")
print("=" * 65)

print("Selected temperature:", SELECTED_T)
print("Current alpha:", 0.25)
print("Seed:", SEED)
print("Output directory:", ALPHA_025_DIR)

# ============================================================
# TRAIN WITH THE ALREADY VERIFIED TRAINING FUNCTION
# ============================================================

result_alpha025 = train_kd_experiment_1598(
    temperature=SELECTED_T,
    alpha=0.25,
    run_dir=ALPHA_025_DIR
)

# ============================================================
# VERIFY SAVED EXPERIMENT
# ============================================================

assert result_alpha025["seed"] == 1598
assert result_alpha025["temperature"] == 5.0
assert result_alpha025["alpha"] == 0.25
assert result_alpha025["epochs_completed"] == 3

best_checkpoint = Path(
    result_alpha025["best_checkpoint"]
)

assert best_checkpoint.is_dir()
assert (best_checkpoint / "config.json").is_file()

assert (
    ALPHA_025_DIR / "training_history.json"
).is_file()

assert (
    ALPHA_025_DIR / "experiment_summary.json"
).is_file()

# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 65)
print("SEED-1598 — ALPHA 0.25 FINAL RESULTS")
print("=" * 65)

print("Temperature:", SELECTED_T)
print("Alpha:", 0.25)
print("Best epoch:", result_alpha025["best_epoch"])

print(
    "Best validation accuracy:",
    f"{result_alpha025['best_validation_accuracy'] * 100:.4f}%"
)

print(
    "Best validation macro-F1:",
    f"{result_alpha025['best_validation_macro_f1']:.6f}"
)

print("Best checkpoint:", best_checkpoint)

print("\n✅ Alpha=0.25 experiment completed.")
print("✅ Three epoch checkpoints saved.")
print("✅ Validation-based selection verified.")
print("✅ Original experiment files untouched.")

print("=" * 65)


SEED-1598 — ALPHA SWEEP
Selected temperature: 5.0
Current alpha: 0.25
Seed: 1598
Output directory: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha025


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



SEED-1598 — KNOWLEDGE DISTILLATION
Temperature: 5.0
Alpha: 0.25
Seed: 1598
Epochs: 3
GPU: NVIDIA L4
Output: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha025


T=5.0 Epoch 1/3:   0%|          | 0/6750 [00:00<?, ?it/s]

T=5.0 Validation 1:   0%|          | 0/750 [00:00<?, ?it/s]

... (6 outputs left out)

T=5.0 Epoch 3/3:   0%|          | 0/6750 [00:00<?, ?it/s]

T=5.0 Validation 3:   0%|          | 0/750 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


-----------------------------------------------------------------
T=5.0, alpha=0.25, Epoch 3
-----------------------------------------------------------------
Train KD loss: 0.222822
Validation accuracy: 94.79%
Validation macro-F1: 0.9480
Best epoch so far: 3

EXPERIMENT COMPLETE
Temperature: 5.0
Alpha: 0.25
Seed: 1598
Best epoch: 3
Best validation accuracy: 94.79%
Best validation macro-F1: 0.9480
Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha025/checkpoint_epoch_3
Experiment folder: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha025

SEED-1598 — ALPHA 0.25 FINAL RESULTS
Temperature: 5.0
Alpha: 0.25
Best epoch: 3
Best validation accuracy: 94.7917%
Best validation macro-F1: 0.947965
Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha025/checkpoint_epoch_3

✅ Alpha=0.25 experiment completed.
✅ Three epoch checkpoints saved.
✅ Validation-base

In [ ]:

import json
from pathlib import Path

# ============================================================
# STEP 20 — FINAL ALPHA SWEEP EXPERIMENT
# ============================================================

assert SID4 == SEED == 1598

# Read the selected temperature from saved evidence.
selection_path = (
    NEW_PART2 / "selected_temperature.json"
)

assert selection_path.is_file()

selection = json.loads(
    selection_path.read_text(encoding="utf-8")
)

assert selection["seed"] == 1598
assert selection["selected_temperature"] == 5

SELECTED_T = 5.0
SELECTED_ALPHA = 0.75

# ============================================================
# 1. DEFINE PROTECTED OUTPUT DIRECTORY
# ============================================================

ALPHA_075_DIR = (
    NEW_PART2 / "alpha_T5_alpha075"
)

assert ALPHA_075_DIR.resolve().parent == (
    NEW_PART2.resolve()
)

assert (
    not ALPHA_075_DIR.exists()
    or not any(ALPHA_075_DIR.iterdir())
), (
    "Alpha=0.75 experiment directory contains files. "
    "Stop to avoid overwriting previous work."
)

print("=" * 65)
print("SEED-1598 — FINAL ALPHA EXPERIMENT")
print("=" * 65)

print("Temperature:", SELECTED_T)
print("Alpha:", SELECTED_ALPHA)
print("Seed:", SEED)
print("Epochs:", TRAIN_CONFIG_1598["epochs"])
print("Output:", ALPHA_075_DIR)

# ============================================================
# 2. TRAIN USING THE VERIFIED SHARED FUNCTION
# ============================================================

result_alpha075 = train_kd_experiment_1598(
    temperature=SELECTED_T,
    alpha=SELECTED_ALPHA,
    run_dir=ALPHA_075_DIR
)

# ============================================================
# 3. VERIFY THE COMPLETED EXPERIMENT
# ============================================================

assert result_alpha075["seed"] == 1598
assert result_alpha075["temperature"] == 5.0
assert result_alpha075["alpha"] == 0.75
assert result_alpha075["epochs_completed"] == 3

best_checkpoint = Path(
    result_alpha075["best_checkpoint"]
)

assert best_checkpoint.is_dir()
assert (best_checkpoint / "config.json").is_file()

assert (
    ALPHA_075_DIR / "training_history.json"
).is_file()

assert (
    ALPHA_075_DIR / "experiment_summary.json"
).is_file()

for epoch in range(1, 4):
    epoch_dir = (
        ALPHA_075_DIR / f"checkpoint_epoch_{epoch}"
    )

    assert epoch_dir.is_dir()
    assert (epoch_dir / "config.json").is_file()

# ============================================================
# 4. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 65)
print("SEED-1598 — ALPHA 0.75 FINAL RESULTS")
print("=" * 65)

print("Temperature:", SELECTED_T)
print("Alpha:", SELECTED_ALPHA)
print("Best epoch:", result_alpha075["best_epoch"])

print(
    "Best validation accuracy:",
    f"{result_alpha075['best_validation_accuracy'] * 100:.4f}%"
)

print(
    "Best validation macro-F1:",
    f"{result_alpha075['best_validation_macro_f1']:.6f}"
)

print("Best checkpoint:", best_checkpoint)

print("\n✅ Alpha=0.75 training completed.")
print("✅ All epoch checkpoints saved.")
print("✅ Entire alpha sweep training completed.")
print("✅ Original seed-42 experiments untouched.")
print("=" * 65)


SEED-1598 — FINAL ALPHA EXPERIMENT
Temperature: 5.0
Alpha: 0.75
Seed: 1598
Epochs: 3
Output: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha075


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



SEED-1598 — KNOWLEDGE DISTILLATION
Temperature: 5.0
Alpha: 0.75
Seed: 1598
Epochs: 3
GPU: NVIDIA L4
Output: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha075


T=5.0 Epoch 1/3:   0%|          | 0/6750 [00:00<?, ?it/s]

T=5.0 Validation 1:   0%|          | 0/750 [00:00<?, ?it/s]

... (6 outputs left out)

T=5.0 Epoch 3/3:   0%|          | 0/6750 [00:00<?, ?it/s]

T=5.0 Validation 3:   0%|          | 0/750 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


-----------------------------------------------------------------
T=5.0, alpha=0.75, Epoch 3
-----------------------------------------------------------------
Train KD loss: 0.154398
Validation accuracy: 94.66%
Validation macro-F1: 0.9467
Best epoch so far: 3

EXPERIMENT COMPLETE
Temperature: 5.0
Alpha: 0.75
Seed: 1598
Best epoch: 3
Best validation accuracy: 94.66%
Best validation macro-F1: 0.9467
Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha075/checkpoint_epoch_3
Experiment folder: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha075

SEED-1598 — ALPHA 0.75 FINAL RESULTS
Temperature: 5.0
Alpha: 0.75
Best epoch: 3
Best validation accuracy: 94.6583%
Best validation macro-F1: 0.946652
Best checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha075/checkpoint_epoch_3

✅ Alpha=0.75 training completed.
✅ All epoch checkpoints saved.
✅ Entire alpha sweep 

In [4]:

import json
import math
from pathlib import Path
import pandas as pd

# ============================================================
# 1. VERIFY SEED AND SELECTED TEMPERATURE
# ============================================================

assert SID4 == SEED == 1598

selection_file = NEW_PART2 / "selected_temperature.json"
assert selection_file.is_file()

temperature_selection = json.loads(
    selection_file.read_text(encoding="utf-8")
)

assert temperature_selection["seed"] == 1598
assert temperature_selection["selected_temperature"] == 5

# ============================================================
# 2. READ THE THREE ACTUAL ALPHA EXPERIMENTS
# ============================================================

ALPHA_RUNS = {
    0.25: NEW_PART2 / "alpha_T5_alpha025",
    0.50: NEW_PART2 / "temperature_T5_alpha05",
    0.75: NEW_PART2 / "alpha_T5_alpha075",
}

rows = []

for alpha, run_dir in ALPHA_RUNS.items():

    summary_path = run_dir / "experiment_summary.json"
    config_path = run_dir / "run_config.json"

    assert summary_path.is_file(), summary_path
    assert config_path.is_file(), config_path

    summary = json.loads(
        summary_path.read_text(encoding="utf-8")
    )

    config = json.loads(
        config_path.read_text(encoding="utf-8")
    )

    assert summary["seed"] == 1598
    assert math.isclose(summary["temperature"], 5.0)
    assert math.isclose(summary["alpha"], alpha)
    assert summary["epochs_completed"] == 3
    assert len(summary["history"]) == 3

    assert config["seed"] == 1598
    assert config["batch_size"] == 16
    assert config["epochs"] == 3
    assert config["alpha_definition"] == "cross_entropy_weight"

    checkpoint = Path(summary["best_checkpoint"])

    assert checkpoint.is_dir()
    assert (checkpoint / "config.json").is_file()

    rows.append({
        "temperature": 5.0,
        "alpha": alpha,
        "seed": 1598,
        "best_epoch": int(summary["best_epoch"]),
        "validation_accuracy": float(
            summary["best_validation_accuracy"]
        ),
        "validation_macro_f1": float(
            summary["best_validation_macro_f1"]
        ),
        "best_checkpoint": str(checkpoint)
    })

# ============================================================
# 3. SELECT USING VALIDATION ONLY
# ============================================================

alpha_results = pd.DataFrame(rows).sort_values(
    "alpha"
).reset_index(drop=True)

ranked = alpha_results.sort_values(
    ["validation_accuracy", "validation_macro_f1"],
    ascending=[False, False]
)

winner = ranked.iloc[0]

BEST_T_FINAL = float(winner["temperature"])
BEST_ALPHA_FINAL = float(winner["alpha"])
BEST_STUDENT_CHECKPOINT = str(winner["best_checkpoint"])

final_selection = {
    "seed": 1598,
    "selected_temperature": BEST_T_FINAL,
    "selected_alpha": BEST_ALPHA_FINAL,
    "alpha_definition": "cross_entropy_weight",
    "selection_dataset": "AG News validation",
    "selection_metric": "validation_accuracy",
    "tie_breaker": "validation_macro_f1",
    "best_epoch": int(winner["best_epoch"]),
    "validation_accuracy": float(
        winner["validation_accuracy"]
    ),
    "validation_macro_f1": float(
        winner["validation_macro_f1"]
    ),
    "best_checkpoint": BEST_STUDENT_CHECKPOINT,
    "test_set_used_for_selection": False
}

# ============================================================
# 4. SAVE WITHOUT SILENT OVERWRITING
# ============================================================

comparison_path = NEW_PART2 / "alpha_sweep_comparison.csv"
final_path = NEW_PART2 / "selected_final_student.json"

def save_text_safely(path, content):
    if path.exists():
        assert path.read_text(encoding="utf-8") == content, (
            f"Existing file differs: {path}. "
            "Inspect before overwriting."
        )
        print("Verified existing:", path)
    else:
        path.write_text(content, encoding="utf-8")
        print("Saved:", path)

save_text_safely(
    comparison_path,
    alpha_results.to_csv(index=False)
)

save_text_safely(
    final_path,
    json.dumps(final_selection, indent=2)
)

# ============================================================
# 5. DISPLAY RESULTS
# ============================================================

print("=" * 65)
print("SEED-1598 — FINAL ALPHA SWEEP")
print("=" * 65)

display(alpha_results[
    [
        "alpha",
        "best_epoch",
        "validation_accuracy",
        "validation_macro_f1"
    ]
])

print("\nSELECTED FINAL STUDENT")
print("Temperature:", BEST_T_FINAL)
print("Alpha:", BEST_ALPHA_FINAL)
print("Best epoch:", final_selection["best_epoch"])

print(
    "Validation accuracy:",
    f"{final_selection['validation_accuracy'] * 100:.4f}%"
)

print(
    "Validation macro-F1:",
    f"{final_selection['validation_macro_f1']:.6f}"
)

print("Checkpoint:", BEST_STUDENT_CHECKPOINT)

print("\n All alpha experiments verified.")
print(" Student selected using validation only.")
print(" Final checkpoint exists.")
print(" No additional training performed.")
print(" Test set not used for selection.")

print("=" * 65)


Saved: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_sweep_comparison.csv
Saved: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/selected_final_student.json
SEED-1598 — FINAL ALPHA SWEEP


,alpha,best_epoch,validation_accuracy,validation_macro_f1
0,0.25,3,0.947917,0.947965
1,0.50,3,0.947750,0.947808
2,0.75,3,0.946583,0.946652



SELECTED FINAL STUDENT
Temperature: 5.0
Alpha: 0.25
Best epoch: 3
Validation accuracy: 94.7917%
Validation macro-F1: 0.947965
Checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha025/checkpoint_epoch_3

 All alpha experiments verified.
 Student selected using validation only.
 Final checkpoint exists.
 No additional training performed.
 Test set not used for selection.


**Loss-weight sweep (validation only, T=5):** α=0.25 → **94.7917%** (macro-F1 0.947965); α=0.50 → 94.7750%; α=0.75 → 94.6583%. Winner: `alpha_T5_alpha025/checkpoint_epoch_3`. Selection did not use the test set.

## 08 | Final student and held-out AG News evaluation

Load the selected SEED=1598 checkpoint and evaluate BERT and DistilBERT on the same untouched 7,600 AG News test records. No test result was used to choose T or α.

In [5]:

import json
import torch
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Read the actual final selection
selection_path = NEW_PART2 / "selected_final_student.json"

selection = json.loads(
    selection_path.read_text(encoding="utf-8")
)

assert selection["seed"] == 1598
assert selection["selected_temperature"] == 5.0
assert selection["selected_alpha"] == 0.25
assert selection["test_set_used_for_selection"] is False

# Recover the saved checkpoint
checkpoint = Path(selection["best_checkpoint"])

assert checkpoint.is_dir()
assert (checkpoint / "config.json").is_file()
assert (checkpoint / "model.safetensors").is_file()

# Use the official DistilBERT tokenizer
student_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-uncased"
)

# Load the actual trained weights, not fresh weights
final_student = AutoModelForSequenceClassification.from_pretrained(
    str(checkpoint),
    local_files_only=True
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

final_student = final_student.to(DEVICE)
final_student.eval()

assert final_student.config.num_labels == 4

# Verify a forward pass
sample = student_tokenizer(
    "Global markets reacted to the latest economic news.",
    return_tensors="pt",
    truncation=True,
    max_length=256
)

sample = {
    k: v.to(DEVICE)
    for k, v in sample.items()
}

with torch.inference_mode():
    logits = final_student(**sample).logits

assert logits.shape == (1, 4)
assert torch.isfinite(logits).all()

print("=" * 60)
print("FINAL STUDENT CHECKPOINT VERIFICATION")
print("=" * 60)
print("Seed:", selection["seed"])
print("Temperature:", selection["selected_temperature"])
print("Alpha:", selection["selected_alpha"])
print("Checkpoint:", checkpoint)
print("Device:", DEVICE)
print("Number of labels:", final_student.config.num_labels)
print("Output logits shape:", tuple(logits.shape))
print("Validation accuracy:", selection["validation_accuracy"])
print()
print(" Saved model loaded successfully.")
print(" Forward pass successful.")
print(" No training performed.")
print(" Test set not accessed.")
print("=" * 60)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

FINAL STUDENT CHECKPOINT VERIFICATION
Seed: 1598
Temperature: 5.0
Alpha: 0.25
Checkpoint: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/alpha_T5_alpha025/checkpoint_epoch_3
Device: cuda
Number of labels: 4
Output logits shape: (1, 4)
Validation accuracy: 0.9479166666666666

✅ Saved model loaded successfully.
✅ Forward pass successful.
✅ No training performed.
✅ Test set not accessed.


In [6]:



import json
import gc
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

assert SEED == SID4 == 1598

selection = json.loads(
    (NEW_PART2 / "selected_final_student.json").read_text()
)
assert selection["selected_temperature"] == 5
assert selection["selected_alpha"] == 0.25
assert selection["test_set_used_for_selection"] is False

TEACHER_DIR = ROOT / "teacher_model"
STUDENT_DIR = Path(selection["best_checkpoint"])

FINAL_DIR = NEW_PART2 / "final_test_evaluation"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["World", "Sports", "Business", "Sci/Tech"]
DEVICE = torch.device("cuda")

# ------------------------------------------------------------
# 1. Load the untouched official AG News test partition
# ------------------------------------------------------------

dataset = load_dataset("fancyzhx/ag_news")
test_data = dataset["test"]

assert len(test_data) == 7600

true_labels = np.asarray(test_data["label"])
texts = test_data["text"]

print("Test examples:", len(test_data))
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 2. Evaluate a saved model without training
# ------------------------------------------------------------

def evaluate_checkpoint(model_dir, tokenizer_dir, name):

    tokenizer = AutoTokenizer.from_pretrained(
        str(tokenizer_dir)
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        str(model_dir),
        local_files_only=True
    ).to(DEVICE)

    model.eval()
    assert model.config.num_labels == 4

    predictions = []

    with torch.inference_mode():

        for start in tqdm(
            range(0, len(texts), 16),
            desc=name
        ):
            batch_texts = texts[start:start+16]

            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt"
            )

            inputs = {
                k: v.to(DEVICE)
                for k, v in inputs.items()
            }

            logits = model(**inputs).logits

            predictions.extend(
                logits.argmax(dim=-1).cpu().tolist()
            )

    predictions = np.asarray(predictions)

    assert len(predictions) == 7600

    accuracy = accuracy_score(
        true_labels, predictions
    )

    macro_f1 = f1_score(
        true_labels,
        predictions,
        average="macro"
    )

    report = classification_report(
        true_labels,
        predictions,
        labels=[0,1,2,3],
        target_names=LABELS,
        output_dict=True,
        zero_division=0
    )

    print(f"\n{name}")
    print(f"Accuracy: {accuracy*100:.4f}%")
    print(f"Macro-F1: {macro_f1:.6f}")

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return predictions, {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "classification_report": report
    }

# Free the earlier loaded student before evaluation
del final_student
gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. Teacher evaluation
# ------------------------------------------------------------

teacher_preds, teacher_metrics = evaluate_checkpoint(
    TEACHER_DIR,
    TEACHER_DIR,
    "BERT Teacher"
)

# ------------------------------------------------------------
# 4. Corrected student evaluation
# ------------------------------------------------------------

student_preds, student_metrics = evaluate_checkpoint(
    STUDENT_DIR,
    "distilbert/distilbert-base-uncased",
    "DistilBERT Student — SEED 1598"
)

# ------------------------------------------------------------
# 5. Save paired predictions for later error analysis
# ------------------------------------------------------------

results = pd.DataFrame({
    "test_index": np.arange(7600),
    "text": texts,
    "true_label": true_labels,
    "teacher_prediction": teacher_preds,
    "student_prediction": student_preds
})

results["teacher_correct"] = (
    results["teacher_prediction"] == results["true_label"]
)

results["student_correct"] = (
    results["student_prediction"] == results["true_label"]
)

results.to_csv(
    FINAL_DIR / "paired_test_predictions.csv",
    index=False
)

metrics = {
    "seed": 1598,
    "temperature": 5.0,
    "alpha": 0.25,
    "test_examples": 7600,
    "max_length": 256,
    "batch_size": 16,
    "teacher": teacher_metrics,
    "student": student_metrics
}

(FINAL_DIR / "final_test_metrics.json").write_text(
    json.dumps(metrics, indent=2)
)

# ------------------------------------------------------------
# 6. Display final accuracy comparison
# ------------------------------------------------------------

print("\n" + "="*65)
print("FINAL AG NEWS TEST COMPARISON — SEED 1598")
print("="*65)

print(
    f"Teacher Accuracy: "
    f"{teacher_metrics['accuracy']*100:.4f}%"
)
print(
    f"Student Accuracy: "
    f"{student_metrics['accuracy']*100:.4f}%"
)

print(
    f"Teacher Macro-F1: "
    f"{teacher_metrics['macro_f1']:.6f}"
)
print(
    f"Student Macro-F1: "
    f"{student_metrics['macro_f1']:.6f}"
)

print(
    "Accuracy difference (student - teacher):",
    f"{(student_metrics['accuracy']-teacher_metrics['accuracy'])*100:+.4f}",
    "percentage points"
)

print("\nSaved results to:", FINAL_DIR)
print(" Same 7600 test examples.")
print(" Same GPU and evaluation settings.")
print(" Corrected student checkpoint.")
print(" No retraining.")
print("="*65)


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

... (4 outputs left out)


BERT Teacher
Accuracy: 94.4605%
Macro-F1: 0.944652


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT Student — SEED 1598:   0%|          | 0/475 [00:00<?, ?it/s]


DistilBERT Student — SEED 1598
Accuracy: 94.6579%
Macro-F1: 0.946626

FINAL AG NEWS TEST COMPARISON — SEED 1598
Teacher Accuracy: 94.4605%
Student Accuracy: 94.6579%
Teacher Macro-F1: 0.944652
Student Macro-F1: 0.946626
Accuracy difference (student - teacher): +0.1974 percentage points

Saved results to: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/final_test_evaluation
 Same 7600 test examples.
 Same GPU and evaluation settings.
 Corrected student checkpoint.
 No retraining.


## 09 | Per-class accuracy and paired mistakes

Compare per-class accuracy and extract five cases where the teacher is correct and the student is wrong, using the saved paired predictions.

In [7]:



import pandas as pd
import json
import numpy as np

FINAL_DIR = (
    NEW_PART2 / "final_test_evaluation"
)

df = pd.read_csv(
    FINAL_DIR / "paired_test_predictions.csv"
)

assert len(df) == 7600
assert df["test_index"].is_unique
assert not df[[
    "true_label",
    "teacher_prediction",
    "student_prediction"
]].isna().any().any()

LABEL_NAMES = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

# ------------------------------------------------------------
# 1. Class-wise accuracy comparison
# ------------------------------------------------------------

rows = []

for label_id, label_name in LABEL_NAMES.items():

    subset = df[df["true_label"] == label_id]

    teacher_acc = (
        subset["teacher_prediction"]
        == subset["true_label"]
    ).mean()

    student_acc = (
        subset["student_prediction"]
        == subset["true_label"]
    ).mean()

    rows.append({
        "class": label_name,
        "samples": len(subset),
        "teacher_accuracy_pct": teacher_acc * 100,
        "student_accuracy_pct": student_acc * 100,
        "student_minus_teacher_pp":
            (student_acc - teacher_acc) * 100
    })

class_results = pd.DataFrame(rows)

print("=" * 65)
print("PER-CLASS TEST ACCURACY — SEED 1598")
print("=" * 65)

display(class_results.round(4))

class_results.to_csv(
    FINAL_DIR / "per_class_accuracy.csv",
    index=False
)

# ------------------------------------------------------------
# 2. Class with greatest degradation
# ------------------------------------------------------------

worst_row = class_results.loc[
    class_results["student_minus_teacher_pp"].idxmin()
]

worst_class = worst_row["class"]

print("\nClass with greatest student-minus-teacher drop:")
print(worst_class)
print(
    "Difference:",
    f"{worst_row['student_minus_teacher_pp']:+.4f}",
    "percentage points"
)

if worst_row["student_minus_teacher_pp"] >= 0:
    print(
        "Note: No class has a negative net accuracy change."
    )

# ------------------------------------------------------------
# 3. Teacher correct, student wrong
# ------------------------------------------------------------

failures = df[
    (df["teacher_prediction"] == df["true_label"])
    & (df["student_prediction"] != df["true_label"])
].copy()

failures["true_class"] = (
    failures["true_label"].map(LABEL_NAMES)
)

failures["student_predicted_class"] = (
    failures["student_prediction"].map(LABEL_NAMES)
)

worst_failures = failures[
    failures["true_class"] == worst_class
]

# Prefer examples from the class with greatest drop
if len(worst_failures) >= 5:
    examples = worst_failures.head(5).copy()
else:
    examples = pd.concat([
        worst_failures,
        failures[
            ~failures["test_index"].isin(
                worst_failures["test_index"]
            )
        ].head(5 - len(worst_failures))
    ])

assert len(examples) == 5

print("\n" + "=" * 65)
print("FIVE TEACHER-CORRECT / STUDENT-WRONG EXAMPLES")
print("=" * 65)

for number, (_, row) in enumerate(
    examples.iterrows(), start=1
):
    print(f"\nEXAMPLE {number}")
    print("Test index:", row["test_index"])
    print("Actual class:", row["true_class"])
    print(
        "Student prediction:",
        row["student_predicted_class"]
    )
    print("Article text:")
    print(row["text"])
    print("-" * 65)

failures.to_csv(
    FINAL_DIR / "all_teacher_correct_student_wrong.csv",
    index=False
)

examples.to_csv(
    FINAL_DIR / "five_failure_examples.csv",
    index=False
)

print("\nTotal teacher-correct/student-wrong:",
      len(failures))

print("\nFiles saved to:", FINAL_DIR)
print(" Per-class accuracy calculated.")
print(" Five genuine paired errors extracted.")
print(" No training performed.")
print("=" * 65)


PER-CLASS TEST ACCURACY — SEED 1598


,class,samples,teacher_accuracy_pct,student_accuracy_pct,student_minus_teacher_pp
0,World,1900,94.9474,95.4211,0.4737
1,Sports,1900,98.8421,98.8947,0.0526
2,Business,1900,91.3684,91.2105,-0.1579
3,Sci/Tech,1900,92.6842,93.1053,0.4211



Class with greatest student-minus-teacher drop:
Business
Difference: -0.1579 percentage points

FIVE TEACHER-CORRECT / STUDENT-WRONG EXAMPLES

EXAMPLE 1
Test index: 110
Actual class: Business
Student prediction: Sci/Tech
Article text:
Yahoo! Ups Ante for Small Businesses Web giant Yahoo! is gambling that price cuts on its domain name registration and Web hosting products will make it more competitive with discounters in the space -- which means that small businesses looking to move online get a sweeter deal through ...
-----------------------------------------------------------------

EXAMPLE 2
Test index: 399
Actual class: Business
Student prediction: Sci/Tech
Article text:
Nortel Downsizes Again Aug. 23, 2004 (TheDeal.com) Problem-plagued Nortel Networks Corp. announced plans Thursday, Aug. 19, to eliminate an additional 3,500 jobs and fire seven more senior executives as the company labors to reinvent 
-----------------------------------------------------------------

EXAMPLE 3
Tes

### Interpretation: where the student makes mistakes

| Class | Teacher accuracy | Student accuracy | Student − teacher (pp) |
|---|---:|---:|---:|
| World | 94.9474% | 95.4211% | +0.4737 |
| Sports | 98.8421% | 98.8947% | +0.0526 |
| **Business** | **91.3684%** | **91.2105%** | **−0.1579** |
| Sci/Tech | 92.6842% | 93.1053% | +0.4211 |

The **Business** class has the largest decrease (−0.1579 percentage points). In the five saved teacher-correct/student-wrong examples, the student predicts **Sci/Tech** four times and **World** once. Many are business stories mentioning web hosting, technology-company restructuring, enterprise hardware or online advertising. The student appears to confuse these overlapping subject areas; this is an interpretation of the examples, not a verified explanation of internal model attention. There are **92** teacher-correct/student-wrong cases in the full test set.

## 10 | Fair teacher–student latency and size benchmark

Benchmark both models on Tesla T4, batch size 1, padded sequence length 256, 20 warmups, 100 timed runs. Model forward-pass latency excludes tokenization.

In [8]:
import gc
import json
import time
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

assert torch.cuda.is_available()

DEVICE = torch.device("cuda")
TEACHER_DIR = ROOT / "teacher_model"

selection = json.loads(
    (NEW_PART2 / "selected_final_student.json").read_text()
)

STUDENT_DIR = Path(selection["best_checkpoint"])
FINAL_DIR = NEW_PART2 / "final_test_evaluation"

BATCH_SIZE = 1
MAX_LENGTH = 256
WARMUP = 20
REPEATS = 100

# Same real AG News inputs for both models
benchmark_texts = list(texts[:REPEATS])

assert len(benchmark_texts) == REPEATS

def checkpoint_size_mb(directory):
    directory = Path(directory)

    # Count stored checkpoint files, not arbitrary output folders
    return sum(
        p.stat().st_size
        for p in directory.rglob("*")
        if p.is_file()
    ) / (1024 ** 2)

def run_benchmark(name, model_dir, tokenizer_dir):

    gc.collect()
    torch.cuda.empty_cache()

    tokenizer = AutoTokenizer.from_pretrained(
        str(tokenizer_dir)
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        str(model_dir),
        local_files_only=True
    ).to(DEVICE)

    model.eval()

    # Tokenize before timing.
    # Fixed padding ensures comparable tensor shapes.
    batches = [
        tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )
        for text in benchmark_texts
    ]

    batches = [
        {k: v.to(DEVICE) for k, v in batch.items()}
        for batch in batches
    ]

    # Warm up CUDA kernels
    with torch.inference_mode():
        for i in range(WARMUP):
            model(**batches[i % len(batches)])

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    latencies_ms = []

    with torch.inference_mode():
        for batch in batches:
            torch.cuda.synchronize()
            start = time.perf_counter()

            model(**batch)

            torch.cuda.synchronize()
            elapsed_ms = (
                time.perf_counter() - start
            ) * 1000

            latencies_ms.append(elapsed_ms)

    peak_gb = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    result = {
        "model": name,
        "gpu": torch.cuda.get_device_name(0),
        "batch_size": BATCH_SIZE,
        "sequence_length": MAX_LENGTH,
        "warmup_runs": WARMUP,
        "timed_runs": REPEATS,
        "p50_latency_ms": float(
            np.percentile(latencies_ms, 50)
        ),
        "p95_latency_ms": float(
            np.percentile(latencies_ms, 95)
        ),
        "peak_gpu_memory_gb": float(peak_gb),
        "checkpoint_size_mb": float(
            checkpoint_size_mb(model_dir)
        )
    }

    del model, batches
    gc.collect()
    torch.cuda.empty_cache()

    return result

teacher_benchmark = run_benchmark(
    "BERT Teacher",
    TEACHER_DIR,
    TEACHER_DIR
)

student_benchmark = run_benchmark(
    "DistilBERT Student",
    STUDENT_DIR,
    "distilbert/distilbert-base-uncased"
)

comparison = pd.DataFrame([
    teacher_benchmark,
    student_benchmark
])

display(comparison.round(4))

comparison.to_csv(
    FINAL_DIR / "teacher_student_benchmark.csv",
    index=False
)

(FINAL_DIR / "benchmark_config.json").write_text(
    json.dumps({
        "seed": 1598,
        "student_temperature": 5,
        "student_alpha": 0.25,
        "benchmark_method": "synchronized wall-clock forward pass",
        "tokenization_timed": False,
        "memory_metric": "torch CUDA peak allocated memory",
        "disk_metric": "checkpoint directory size",
        "results": comparison.to_dict(orient="records")
    }, indent=2)
)

print("\n" + "=" * 65)
print("FAIR TEACHER / STUDENT BENCHMARK COMPLETE")
print("=" * 65)

print("GPU:", torch.cuda.get_device_name(0))
print("Batch size:", BATCH_SIZE)
print("Sequence length:", MAX_LENGTH)
print("Warmup iterations:", WARMUP)
print("Timed iterations:", REPEATS)
print("Files saved to:", FINAL_DIR)
print(" Same hardware and input examples.")
print(" p50 and p95 measured.")
print(" Peak allocated GPU memory measured.")
print(" Checkpoint directory sizes measured.")
print(" No retraining.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

,model,gpu,batch_size,sequence_length,warmup_runs,timed_runs,p50_latency_ms,p95_latency_ms,peak_gpu_memory_gb,checkpoint_size_mb
0,BERT Teacher,Tesla T4,1,256,20,100,16.7124,17.1056,0.4264,418.5823
1,DistilBERT Student,Tesla T4,1,256,20,100,8.5814,8.9077,0.2680,255.4317



FAIR TEACHER / STUDENT BENCHMARK COMPLETE
GPU: Tesla T4
Batch size: 1
Sequence length: 256
Warmup iterations: 20
Timed iterations: 100
Files saved to: /content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/final_test_evaluation
 Same hardware and input examples.
 p50 and p95 measured.
 Peak allocated GPU memory measured.
 Checkpoint directory sizes measured.
 No retraining.


**Benchmark conditions:** same Tesla T4 GPU, same 100 AG News texts, batch size 1, maximum/padded sequence length 256, 20 warmups, 100 timed forward passes, CUDA synchronization. Disk size is the checkpoint *directory* size. We did not time tokenization or model loading. Relative to the teacher: p50 latency reduction ≈48.65%, memory reduction ≈37.15%, disk reduction ≈38.98%.

## 11 | Complete teacher logits for all 120,000 training records

Post-training compliance step: reuse the existing 108,000 logits and generate the 12,000 held-out validation logits, restoring original AG News index order. This does not use validation examples for student training.

In [16]:

import os
import gc
import json
import hashlib

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from tqdm.auto import tqdm


# ============================================================
# 1. CONFIGURATION AND VERIFIED GOOGLE DRIVE PATHS
# ============================================================

SID4 = 1598
SEED = 1598

ROOT = Path(
    "/content/drive/MyDrive/DATA266_HW12"
)

NEW_PART2 = (
    ROOT / "outputs" / "part2_distillation_seed1598"
)

TEACHER_DIR = ROOT / "teacher_model"

LOGIT_DIR = NEW_PART2 / "teacher_logits"

MANIFEST_PATH = (
    NEW_PART2 / "data_split_manifest.json"
)

OLD_METADATA = (
    LOGIT_DIR / "teacher_logits_metadata.json"
)

OLD_LOGITS = (
    LOGIT_DIR / "teacher_train_logits.npy"
)

FULL_LOGITS = (
    LOGIT_DIR / "teacher_all_120000_logits.npy"
)

FULL_METADATA = (
    LOGIT_DIR / "teacher_all_120000_metadata.json"
)

assert SID4 == SEED == 1598

assert ROOT.is_dir(), "Project root not found"
assert NEW_PART2.is_dir(), "Distillation folder missing"
assert TEACHER_DIR.is_dir(), "Teacher model missing"
assert LOGIT_DIR.is_dir(), "Teacher logits folder missing"

assert MANIFEST_PATH.is_file()
assert OLD_METADATA.is_file()
assert OLD_LOGITS.is_file()

assert torch.cuda.is_available(), (
    "GPU unavailable. Enable GPU in Colab runtime."
)

DEVICE = torch.device("cuda")

print("=" * 70)
print("STEP 27B — COMPLETE TEACHER LOGITS")
print("=" * 70)

print("SEED:", SEED)
print("GPU:", torch.cuda.get_device_name(0))
print("Project:", ROOT)


# ============================================================
# 2. VERIFY ORIGINAL DATA SPLIT AND LOGIT METADATA
# ============================================================

manifest = json.loads(
    MANIFEST_PATH.read_text(encoding="utf-8")
)

metadata = json.loads(
    OLD_METADATA.read_text(encoding="utf-8")
)

manifest_hash = hashlib.sha256(
    MANIFEST_PATH.read_bytes()
).hexdigest()

assert manifest["seed"] == 1598

assert manifest["train_size"] == 108000
assert manifest["validation_size"] == 12000

assert metadata["seed"] == 1598
assert metadata["num_examples"] == 108000
assert metadata["num_classes"] == 4
assert metadata["max_length"] == 256

assert metadata["logits_dtype"] == "float32"

assert metadata["split_manifest_sha256"] == manifest_hash

assert Path(
    metadata["teacher_checkpoint"]
).resolve() == TEACHER_DIR.resolve()

train_indices = np.asarray(
    manifest["train_original_indices"],
    dtype=np.int64
)

validation_indices = np.asarray(
    manifest["validation_original_indices"],
    dtype=np.int64
)

assert train_indices.shape == (108000,)
assert validation_indices.shape == (12000,)

all_indices = np.concatenate(
    [train_indices, validation_indices]
)

assert np.array_equal(
    np.sort(all_indices),
    np.arange(120000)
), "Dataset indices are incomplete or duplicated"

print("\n✅ Data split manifest verified.")
print("Training indices:", len(train_indices))
print("Validation indices:", len(validation_indices))


# ============================================================
# 3. LOAD AND PROTECT ORIGINAL 108,000 LOGITS
# ============================================================

original_logits = np.load(
    OLD_LOGITS,
    mmap_mode="r",
    allow_pickle=False
)

assert original_logits.shape == (108000, 4)

assert original_logits.dtype == np.float32

assert np.isfinite(original_logits).all()

print("\n Existing teacher logits verified.")
print("Original logits shape:", original_logits.shape)

# Existing complete outputs must not be overwritten.
if FULL_LOGITS.exists() or FULL_METADATA.exists():

    assert FULL_LOGITS.is_file() and FULL_METADATA.is_file(), (
        "Only one final output file exists. "
        "Inspect the files before proceeding."
    )

    existing_full = np.load(
        FULL_LOGITS,
        mmap_mode="r",
        allow_pickle=False
    )

    existing_meta = json.loads(
        FULL_METADATA.read_text(encoding="utf-8")
    )

    assert existing_full.shape == (120000, 4)
    assert existing_full.dtype == np.float32
    assert np.isfinite(existing_full).all()

    assert np.array_equal(
        existing_full[train_indices],
        original_logits
    )

    assert existing_meta["seed"] == 1598
    assert existing_meta["num_examples"] == 120000
    assert existing_meta["manifest_sha256"] == manifest_hash

    print("\n✅ COMPLETE LOGIT CACHE ALREADY EXISTS.")
    print("Shape:", existing_full.shape)
    print("File:", FULL_LOGITS)
    print("No files overwritten.")

else:

    # ========================================================
    # 4. LOAD ORIGINAL AG NEWS TRAINING DATASET
    # ========================================================

    print("\nLoading official AG News training dataset...")

    dataset = load_dataset(
        "fancyzhx/ag_news"
    )["train"]

    assert len(dataset) == 120000

    print("Dataset loaded:", len(dataset), "articles")


    # ========================================================
    # 5. LOAD FINE-TUNED TEACHER MODEL
    # ========================================================

    print("\nLoading locally saved BERT teacher...")

    tokenizer = AutoTokenizer.from_pretrained(
        str(TEACHER_DIR),
        local_files_only=True
    )

    teacher = (
        AutoModelForSequenceClassification
        .from_pretrained(
            str(TEACHER_DIR),
            local_files_only=True
        )
        .to(DEVICE)
    )

    teacher.eval()

    assert teacher.config.num_labels == 4

    print("✅ Teacher loaded successfully.")
    print("Model device:", DEVICE)


    # ========================================================
    # 6. RESTORE ORIGINAL 108,000 LOGITS
    # ========================================================

    full_logits = np.empty(
        (120000, 4),
        dtype=np.float32
    )

    full_logits[train_indices] = original_logits

    print("\n Original 108,000 logits restored.")
    print("Original AG News ordering maintained.")


    # ========================================================
    # 7. GENERATE MISSING 12,000 VALIDATION LOGITS
    # ========================================================

    BATCH_SIZE = 64
    MAX_LENGTH = 256

    print("\nGenerating logits for missing 12,000 articles...")

    with torch.inference_mode():

        for start in tqdm(
            range(
                0,
                len(validation_indices),
                BATCH_SIZE
            ),
            desc="Generating missing teacher logits"
        ):

            end = min(
                start + BATCH_SIZE,
                len(validation_indices)
            )

            batch_indices = validation_indices[start:end]

            # IMPORTANT TOKENIZER FIX:
            # Explicitly convert Dataset column to list[str].

            batch_texts = list(
                dataset.select(
                    batch_indices.tolist()
                )["text"]
            )

            assert len(batch_texts) == len(batch_indices)

            assert all(
                isinstance(text, str)
                for text in batch_texts
            ), "Tokenizer input must contain strings"

            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt"
            )

            encoded = {
                key: value.to(DEVICE)
                for key, value in encoded.items()
            }

            outputs = teacher(**encoded)

            batch_logits = (
                outputs.logits
                .detach()
                .float()
                .cpu()
                .numpy()
            )

            assert batch_logits.shape == (
                len(batch_indices), 4
            )

            assert np.isfinite(batch_logits).all()

            full_logits[batch_indices] = batch_logits

    torch.cuda.synchronize()

    print("\n✅ All missing validation logits generated.")


    # ========================================================
    # 8. VERIFY COMPLETE LOGIT MATRIX
    # ========================================================

    assert full_logits.shape == (120000, 4)

    assert full_logits.dtype == np.float32

    assert np.isfinite(full_logits).all()

    assert np.array_equal(
        full_logits[train_indices],
        original_logits
    ), "Original 108,000 logits were modified"

    print("\n Complete logit matrix verified.")
    print("Final shape:", full_logits.shape)
    print("Existing rows preserved:", 108000)
    print("New rows generated:", 12000)


    # ========================================================
    # 9. CREATE REPRODUCIBILITY METADATA
    # ========================================================

    new_metadata = {
        "seed": 1598,
        "dataset": "fancyzhx/ag_news",
        "source_split": "train",
        "teacher_checkpoint": str(TEACHER_DIR),
        "num_examples": 120000,
        "num_classes": 4,
        "max_length": MAX_LENGTH,
        "new_logits_batch_size": BATCH_SIZE,
        "dtype": "float32",
        "shape": [120000, 4],
        "original_training_rows_reused": 108000,
        "new_validation_rows_generated": 12000,
        "ordering": "original_AG_News_training_order",
        "manifest_sha256": manifest_hash,
        "student_training_rows": 108000,
        "student_validation_rows": 12000,
        "validation_used_for_student_training": False,
        "generated_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }


    # ========================================================
    # 10. SAVE SAFELY WITHOUT OVERWRITING OLD FILES
    # ========================================================

    temporary_logits = (
        LOGIT_DIR
        / "teacher_all_120000_logits.partial.npy"
    )

    temporary_metadata = (
        LOGIT_DIR
        / "teacher_all_120000_metadata.partial.json"
    )

    assert not temporary_logits.exists(), (
        "Temporary logits file already exists. "
        "Inspect before rerunning."
    )

    assert not temporary_metadata.exists(), (
        "Temporary metadata file already exists."
    )

    print("\nSaving completed logit cache...")

    with temporary_logits.open("wb") as f:
        np.save(f, full_logits)

    temporary_metadata.write_text(
        json.dumps(new_metadata, indent=2),
        encoding="utf-8"
    )

    # ========================================================
    # 11. VERIFY SAVED FILE BEFORE FINALIZING
    # ========================================================

    saved_logits = np.load(
        temporary_logits,
        mmap_mode="r",
        allow_pickle=False
    )

    assert saved_logits.shape == (120000, 4)

    assert saved_logits.dtype == np.float32

    assert np.isfinite(saved_logits).all()

    assert np.array_equal(
        saved_logits[train_indices],
        original_logits
    )

    assert json.loads(
        temporary_metadata.read_text(encoding="utf-8")
    )["num_examples"] == 120000

    # Rename only after all checks pass.
    os.replace(
        temporary_logits,
        FULL_LOGITS
    )

    os.replace(
        temporary_metadata,
        FULL_METADATA
    )

    print("\n Complete logit cache saved successfully.")

    # Free model and GPU memory
    del teacher
    del full_logits
    del saved_logits

    gc.collect()
    torch.cuda.empty_cache()


# ============================================================
# 12. FINAL DISK VERIFICATION
# ============================================================

verified_logits = np.load(
    FULL_LOGITS,
    mmap_mode="r",
    allow_pickle=False
)

verified_metadata = json.loads(
    FULL_METADATA.read_text(encoding="utf-8")
)

assert verified_logits.shape == (120000, 4)

assert np.isfinite(verified_logits).all()

assert np.array_equal(
    verified_logits[train_indices],
    original_logits
)

assert verified_metadata["num_examples"] == 120000

assert verified_metadata["seed"] == 1598


print("\n" + "=" * 70)
print("STEP 27B — FINAL SUCCESS REPORT")
print("=" * 70)

print("SEED:", 1598)
print("Full logit shape:", verified_logits.shape)

print("Original training logits:", 108000)
print("New validation logits:", 12000)
print("Total AG News training examples:", 120000)

print("\nSaved files:")
print(FULL_LOGITS)
print(FULL_METADATA)

print("\n All 120,000 teacher logits available.")
print(" Original 108,000 training logits preserved.")
print(" Correct dataset ordering verified.")
print(" Student training/validation split unchanged.")
print(" Existing checkpoints and results untouched.")
print(" No student retraining performed.")

print("=" * 70)


STEP 27B — COMPLETE TEACHER LOGITS
SEED: 1598
GPU: Tesla T4
Project: /content/drive/MyDrive/DATA266_HW12

✅ Data split manifest verified.
Training indices: 108000
Validation indices: 12000

 Existing teacher logits verified.
Original logits shape: (108000, 4)

Loading official AG News training dataset...
Dataset loaded: 120000 articles

Loading locally saved BERT teacher...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Teacher loaded successfully.
Model device: cuda

 Original 108,000 logits restored.
Original AG News ordering maintained.

Generating logits for missing 12,000 articles...


Generating missing teacher logits:   0%|          | 0/188 [00:00<?, ?it/s]


✅ All missing validation logits generated.

 Complete logit matrix verified.
Final shape: (120000, 4)
Existing rows preserved: 108000
New rows generated: 12000

Saving completed logit cache...

 Complete logit cache saved successfully.

STEP 27B — FINAL SUCCESS REPORT
SEED: 1598
Full logit shape: (120000, 4)
Original training logits: 108000
New validation logits: 12000
Total AG News training examples: 120000

Saved files:
/content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/teacher_logits/teacher_all_120000_logits.npy
/content/drive/MyDrive/DATA266_HW12/outputs/part2_distillation_seed1598/teacher_logits/teacher_all_120000_metadata.json

 All 120,000 teacher logits available.
 Original 108,000 training logits preserved.
 Correct dataset ordering verified.
 Student training/validation split unchanged.
 Existing checkpoints and results untouched.
 No student retraining performed.


**Coverage and leakage note:** `teacher_train_logits.npy` retains the 108,000 original training logits; `teacher_all_120000_logits.npy` is an *additional* complete cache, with 12,000 validation logits generated later. The full cache documents extraction coverage but was **not** used to train the selected student. Training used 108,000 examples; validation used 12,000; the AG News test split contained 7,600 distinct records.

## Submission artifact map

All Drive locations below are relative to `/content/drive/MyDrive/DATA266_HW12/`.

| Artifact | Location |
|---|---|
| Instructor-provided BERT teacher | `teacher_model/` |
| Seed-specific split and training configuration | `outputs/part2_distillation_seed1598/data_split_manifest.json`, `training_config_seed1598.json` |
| Teacher logits: training / full dataset | `outputs/part2_distillation_seed1598/teacher_logits/teacher_train_logits.npy`, `teacher_all_120000_logits.npy` |
| Sweep comparisons and selections | `outputs/part2_distillation_seed1598/{temperature_sweep_comparison.csv,alpha_sweep_comparison.csv,selected_temperature.json,selected_final_student.json}` |
| Final selected student checkpoint | `outputs/part2_distillation_seed1598/alpha_T5_alpha025/checkpoint_epoch_3/` |
| Final test, per-class/error analyses and benchmarks | `outputs/part2_distillation_seed1598/final_test_evaluation/` |
| Distillation summary and recorded run log | Submitted alongside this notebook as `METRICS_DISTILLATION.md` and `RUN_LOG_DISTILLATION.txt` |

**GPTQ is separate.** The already-produced `METRICS_GPTQ.md` and `RUN_LOG_GPTQ_DRAFT.txt` are not overwritten or represented as distillation artifacts.

**Notebook output policy:** original outputs are evidence from execution; they do not certify that a fresh end-to-end replay will work without a clean output workspace and the model/data dependencies.